# UK Exports Predictor

**The question.** Government export Advisors must decide which UK industries to help, in which Markets, and why. They mostly work from what the UK already sells. That hides two kinds of lead: Markets where the UK sells far less of a Product than comparable exporters do, and Markets where a Barrier, such as the lack of a Trade Agreement, holds UK exports down.

**The approach.** A gradient-boosted model learns from global HS6 goods trade, 2012–2024 (about 226 Exporters, 5,200 Products and 142 M Flows), what Flow to expect between any Exporter, Importer and Product, given their fundamentals. Applied to the UK, it estimates **Export Potential** for every Product–Market pair. Export Potential minus Actual Exports is the **Export Gap**, and positive gaps ranked by size are **Opportunities**. Re-predicting with only the trade-policy features changed gives the **Scenario Effect** of a Trade Agreement or Trading Bloc **Scenario**, and pairs that a Scenario would unlock materially are **Blocked Opportunities**. A FastAPI service serves the saved model, and Advisors use it through a Streamlit app.

**What we found** (v2, real data; sections 7 and 10 give the detail):

- **Across all Exporters the model clearly beats a naive market-share baseline** on the untouched 2023 test year: within-Product rank correlation 0.76 against 0.57, and about a twelfth of its deviance.
- **For UK pairs, scored out-of-fold, it does not beat the naive baseline on the headline measure**: 0.51 against 0.52. Its gain is in the *size* of Export Potential (deviance 569 against 2,263), not in the order of Markets.
- **v2 does not beat v1.** Under a rule fixed before 2023 was read, v2 ranks UK Markets less well than v1 did (0.511 against 0.540) while estimating the size of Flows better. The gain ladder associates most of the drop with v2's new training rows (the Exporter scope, the sampling and the exclusions together), not with its new features or its tuning.
- **Large Export Gaps point at growth, but mostly because they sit on large pairs.** Lift is 5.2, but against pairs of similar size it is 0.79.
- **The model overstates what Trade Agreements are associated with.** For EU–Japan it associates +35% with the agreement against an observed difference-in-differences of +2% (−7% to +14%). A PPML gravity benchmark on the same Flows associates a Trade Agreement with +5.1% (−0.4% to +11.0%), against the model's +58%.
- **The Potential Range covers 86% of held-out outcomes against an 80% target**, but its offsets are fixed dollars per band, so every large pair gets the same range, about $22.9m wide.
- **BACI's UK total for 2023 is 16% below HMRC's own figure.** Non-monetary gold and trade HMRC does not attribute to a Product or Market explain about four fifths of the gap; the pairs both sources record agree closely in rank (0.89).

So the tool is useful for *where to look*, as a ranking of leads for a person to check, and is not a forecast of what an agreement or an export push would deliver. The language throughout is associational: "the model associates a Trade Agreement with +X%", never "the agreement causes +X%".

**How to read this notebook.** Each section opens with what it does and why, and closes with a **Conclusion**. All reusable logic lives in the `exportspredictor` package, which the API imports too, so training and serving compute features identically; the notebook only calls it. With the default parameter below, every cell runs in seconds on a tiny fixture world. Real-data results are quoted in the prose from the v2 runs of 17–18 November 2026 (artefact `20261118T082751Z`).

**Course concepts applied, and where the project goes beyond them** (`ML_AI_Concept_Notes.html`):

| Section | Course concept applied | Beyond the notes |
|---|---|---|
| 1. Flows | Exploratory data analysis; data quality audit; descriptive statistics (skew) | Lazy Polars scans over 142 M rows; Enum-coded Parquet |
| 2. Gravity and WDI | Feature engineering; imputing missing values with an explicit rule | Primary-source cross-check of BACI against HMRC OTS |
| 3. Policy features | Feature engineering; data lineage | Pair-year policy features from per-member WTO dates, cross-checked with CEPII Gravity |
| 4. Feature table | Data leakage (preprocessing and target); out-of-fold target encoding; class imbalance; learning curves and the bias–variance trade-off; drop-column importance | Leave-one-out supply and demand; Horvitz–Thompson weighted zero sampling; a learning curve along the *data* axis that sets the sampling rates within a VRAM budget; Relatedness from a product-space proximity |
| 5. Training | Ensembles (gradient boosting); train/validation/test splits; grid and randomised hyperparameter search; early stopping and L2 regularisation; loss vs metric; GPUs and device placement | Tweedie loss; out-of-fold cross-fitting for the UK (ADR 0001); Bayesian optimisation (TPE with a median pruner) instead of a grid; a data iterator for data that does not fit in memory |
| 6. Scenarios | Correlation is not causation; the counterfactual test; linear regression and coefficients | Monotone constraints on every policy feature (ADR 0002), the Applied Tariff included (ADR 0006); a 3-way fixed-effects PPML gravity model as an independent benchmark |
| 7. Evaluation | Baselines first; regression metrics; held-out validation; worst-group evaluation; drift sensitivity; touch the test set once | A ranking metric within Product; Lift with a cluster bootstrap and a size-matched control; difference-in-differences Event Backtests; split conformal prediction with Mondrian bands (ADR 0007); a pre-registered v1/v2 comparison with a paired bootstrap and a gain ladder |
| 8. Explaining | Global vs local explanations; SHAP and actionable explanations; feature importance | Exact TreeSHAP from each pair's own fold model, in plain words |
| 9. Artefacts | Reproducible artefacts; registry versions | Versioned artefact folders the API serves unchanged |
| 10. Governance | Model cards; data lineage; data and algorithmic bias; representation audit; EU AI Act tiers and intended purpose; the UK approach; PSI drift | Bias review by reporting quality (income level), not only by group size; Potential Range coverage by Exporter as a check of exchangeability |

## Parameters

`FIXTURE_MODE` is the notebook's one parameter (the cell is tagged `parameters`, so papermill and similar tools can set it).

- `FIXTURE_MODE = True` (the default) runs every section on a tiny synthetic world in a few seconds. The test suite executes the notebook this way (`tests/test_notebook.py`), so a broken cell fails the build. Cells that need the real downloads, such as the HMRC cross-check and the data-lineage fingerprints, say so and skip.
- `FIXTURE_MODE = False` runs on the real data and **trains nothing**. The models are trained, evaluated and saved by `python -m exportspredictor.pipeline` (a step of `scripts/rebuild.py`, several hours on an RTX 3080), after the ablation, the learning curve and the search (`exportspredictor.experiments` and `exportspredictor.search`). `scripts/rebuild.py` then runs this notebook with `EXPORTSPREDICTOR_FIXTURE_MODE=0`. It **reads and does not build**: the derived gravity, policy and feature tables the build steps wrote, the newest artefact, the experiment loop's saved results, the saved search and experiment results, and the PPML benchmark saved with the artefact (`python -m exportspredictor.ppml`). It fits nothing and rebuilds no table, so a run cannot change a reported number, and it takes a few minutes.


In [ ]:
import os

FIXTURE_MODE = True  # True: the tiny fixture world, in seconds. False: the real data (reads the newest artefact and saved results; trains and builds nothing).

if os.environ.get("EXPORTSPREDICTOR_FIXTURE_MODE") == "0":
    FIXTURE_MODE = False  # scripts/rebuild.py sets this for the real run
FIXTURE_MODE

In [ ]:
import polars as pl

from exportspredictor import artefacts, baci, loop, pipeline, policy, reporting
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir


## 1. The Flows: BACI HS12

Everything the model knows comes from one dataset, so we start by reading it: what it covers, how skewed it is, and what is provisional. The training data is CEPII's BACI, HS 2012 revision, release V202601 (ADR 0003): every non-zero Flow between about 226 Exporters and Importers for about 5,200 Products, 2012–2024. BACI reconciles each country's export report with its partners' import reports, so a UK Flow is a blend of what the UK and the Market declared, not HMRC's figure. Values are converted from thousand USD to USD.

This section is the course's **exploratory data analysis** and **data-quality audit**, applied at a scale where pandas would not fit: the download is converted once from CSV to Parquet with compact types (Enum codes for countries and Products, so `010121` keeps its leading zero), and Polars scans it lazily, so only the columns and years a query needs are read. The conversion peaks at about 2.5 GB of memory.

In fixture mode the same summaries run on the fixture world's Flows.

In [ ]:
if FIXTURE_MODE:
    flows = build_world().flows.lazy()
else:
    # Downloaded and converted by the BACI build step (`baci.prepare`, in scripts/rebuild.py); read here.
    flows = baci.scan_flows()
flows.head(5).collect()

### Coverage and size per year

Row counts per year, with the number of Products, Exporters and Importers that trade, total world value and the UK's share of it. BACI records only strictly positive Flows, so a missing row means zero *or* unreported: the model treats absent pairs as zeros, which matters for poorly reporting countries.

In [ ]:
flow_summary = baci.summarise_flows(flows)
flow_summary

In [ ]:
flows.filter(pl.col("value_usd") > 0).select(
    flows=pl.len(),
    products=pl.col("hs6").n_unique(),
    exporters=pl.col("exporter").n_unique(),
    importers=pl.col("importer").n_unique(),
).collect()

On the real data (V202601) there are about 9.0 M Flows in 2012, rising to 11.8 M in 2023, about 142 M in total. 5,199 of the 5,202 HS 2012 Products trade in a typical year, and 226 Exporters appear each year (227 across all years). The UK's share of world goods exports in BACI drifts down from about 2.4–2.6% in 2012–2016 to about 1.9% in 2021–2023.

Flow values are extremely skewed: the mean UK Flow (about $1.3 M) is roughly 100 times its median (about $14k), which is why the model uses a Tweedie loss rather than squared error on raw values (course: skewed variables favour the median).

In [ ]:
uk_latest = flows.filter((pl.col("exporter").cast(pl.String) == baci.UK) & (pl.col("value_usd") > 0))
uk_latest.group_by("year").agg(
    mean_usd=pl.col("value_usd").mean(),
    median_usd=pl.col("value_usd").median(),
    markets=pl.col("importer").n_unique(),
    products=pl.col("hs6").n_unique(),
).sort("year").collect()

### The provisional 2024 data

BACI's latest year is provisional: Comtrade is downloaded in January, before every country has reported, and CEPII revises it in the next release. The table below compares each year with the one before. On the real data, 2024 has about 5.5% fewer Flows than 2023 while world value is almost flat, consistent with late reporters still missing small Flows. UK exports fall about 8% in 2024, which may partly reflect the same incompleteness. 2024 is the Baseline year, so the app labels it provisional, and the evaluation (test year 2023) never relies on it.

In [ ]:
flow_summary.select(
    "year",
    flows_change=pl.col("flows").pct_change(),
    world_value_change=pl.col("world_value_usd").pct_change(),
    uk_value_change=pl.col("uk_value_usd").pct_change(),
).tail(3)

**Conclusion.** BACI gives a complete, consistent HS 2012 panel of 142 M non-zero Flows, small enough to scan lazily on one machine. Three properties shape every later choice: values are extremely skewed (hence a Tweedie loss in section 5), absent Flows mean zero *or* unreported (hence sampled zeros in section 4 and the reporting-quality bias review in section 10), and 2024 is provisional (hence a 2023 test year and a stability check in section 7). Section 2 checks BACI's UK figures against HMRC's own.

## 2. Gravity and WDI features: coverage

The gravity model of trade (the oldest empirical regularity in trade economics) says Flows grow with the size of both countries and shrink with the distance between them. These are the non-policy fundamentals the model conditions on. This is feature engineering in the course's sense: domain knowledge turned into inputs, built once in the package (`exportspredictor.gravity`) so training and serving share it.

- **Pair features** from CEPII Gravity (V202211): distance between main cities, contiguity, common official language, colonial tie (ever in a colonial or dependency relationship) and common colonizer. They barely change over time, so each pair takes its latest non-null value.
- **Country-year features**: GDP (current USD) and population for every year 2012–2024.

`gravity.build()` downloads what is missing and writes the derived tables; the data rebuild script runs it. It needs BACI's country code file, so it runs after section 1. The cells below run on real data only (`FIXTURE_MODE = False`).

**Country codes.** BACI uses its own numeric codes. All 227 Exporters and Importers in the 2012–2024 Flows map to an ISO3 code, and every one has Gravity pair features. Three mismatches needed a fix:

- BACI 490 "Other Asia, nes" (ISO3 field `S19`) is Taiwan, mapped to `TWN`.
- Saint Barthélemy (`BLM`) and the French Southern and Antarctic Lands (`ATF`) are missing from Gravity. Their pair features come from a stand-in, Guadeloupe and Réunion. Together they carry about $2.6 bn of Flows over 13 years.

**Missing GDP and population: the rule.** For each country and variable:

1. World Bank WDI, the main source.
2. Otherwise the IMF World Economic Outlook. This covers Taiwan (absent from WDI), South Sudan from 2016, Yemen from 2019 and Eritrea to 2019.
3. Otherwise CEPII Gravity's own figures, available up to 2021.
4. Otherwise linear interpolation between the nearest observed years, or the nearest year carried forward or back at the edges. This applies, for example, to Cuba's GDP from 2021 and Syria's from 2023.
5. Otherwise missing. 21 small territories have no GDP in any source, including Anguilla, the Falklands, Gibraltar, North Korea and Saint Helena. Together they account for about 0.1% of world Flow value. XGBoost learns a default direction for missing values at each split, so these rows still train. Imputing a guess would add noise without adding information.

Each value records its source in `gdp_source` / `population_source`, so any figure can be traced. Some pair features are also missing in Gravity: common language for some territories (e.g. Guam, Curaçao), and colonial ties for China–Hong Kong, China–Taiwan and Israel–Palestine. These follow the same rule and stay missing.

In [ ]:
from exportspredictor import gravity
from exportspredictor.paths import data_dir

if not FIXTURE_MODE:
    # Built by the gravity step (`python -m exportspredictor.gravity`, in scripts/rebuild.py); read here, not rebuilt.
    derived = data_dir() / "derived"
    in_flows = pl.concat(
        [flows.select(baci_iso3=pl.col(side).cast(pl.String)) for side in ["exporter", "importer"]]
    ).unique().collect()
    # Only countries that appear in the 2012-2024 Flows matter; BACI's file also lists retired codes.
    country_map = pl.read_parquet(derived / "country_map.parquet").join(in_flows, on="baci_iso3", how="semi")
    country_year = pl.read_parquet(derived / "country_year.parquet").filter(pl.col("iso3").is_in(country_map["iso3"].implode()))
    print("Exporters and Importers in the Flows:", country_map["baci_iso3"].n_unique())
    print("... with no Gravity pair features:", gravity.unresolved(country_map).height)
    print("... with no WDI data:", country_map.filter(~pl.col("in_wdi")).height)
    display(country_map.filter((pl.col("baci_iso3") != pl.col("iso3")) | pl.col("gravity_proxy").is_not_null()))

Where each GDP and population value comes from, and the countries that stay missing:

In [ ]:
if not FIXTURE_MODE:
    display(country_year.group_by("gdp_source").len().sort("len", descending=True))
    display(country_year.group_by("population_source").len().sort("len", descending=True))
    display(country_year.filter(pl.col("gdp_usd").is_null()).group_by("iso3").len().sort("iso3"))

### Data quality: BACI's UK totals against HMRC OTS 2023

BACI's UK Flows are not HMRC's statistics. BACI blends the UK's export report with each Market's import report (converted from CIF to FOB, weighted by how reliable each reporter has been), recasts every year to HS 2012, and works in USD. HMRC's **Overseas Trade Statistics** (OTS) are what the UK itself declared, in £, in that year's Combined Nomenclature (HS 2022 at six digits for 2023), and they exclude goods in transit and customs-warehouse re-exports (`docs/research/baci-hs12-feasibility.md`, section 4). Before trusting BACI as the target, we measure how far the two differ. This is the course's **data quality audit** with an independent primary source, which the notes do not cover.

`hmrc.build()` (a step in `scripts/rebuild.py`) downloads HMRC's 2023 bulk export archive (one fixed-width record per CN8 × country × port × month, 3.2 M records) and the UN Statistics Division's HS 2022 → HS 2012 conversion table. It sums ports and months to HS6 × Market, converts the codes and maps HMRC's country codes to ISO3. Value HMRC does not attribute to a Product (low-value consignments, suppressed commodity codes, chapter-level estimates) or to a Market (suppressed or unspecified countries) is kept with a missing key, so the totals still add up. `hmrc.crosscheck` then compares the result with BACI's UK Flows of 2023 in USD, at the Bank of England's 2023 average rate ($1.2434 per £).

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the HMRC cross-check needs the real downloads (FIXTURE_MODE = False).")
else:
    from exportspredictor import hmrc

    # The allocated OTS table is written by `hmrc.build` (in scripts/rebuild.py); read here, not rebuilt.
    hmrc_check = hmrc.crosscheck(
        hmrc.load(), hmrc.baci_uk_exports(flows, country_map)
    )
    print(hmrc_check["text"])
    display(pl.DataFrame([{key: value for key, value in hmrc_check.items() if key.endswith("_usd")}]))
    display(pl.DataFrame(hmrc_check["by_sector"]))
    display(pl.DataFrame(hmrc_check["by_market"]))
    print(hmrc_check["pairs"])

**The size of the difference.** For 2023, HMRC records **$528.6bn** of UK goods exports and BACI **$443.0bn**: HMRC is **$85.7bn, or 19%, higher** (BACI is 16% below HMRC). Almost all of that has a clear source:

| Part of the difference | USD bn | Why |
|---|---|---|
| Not attributed to a Product | 27.2 | HMRC books low-value consignments to a single code (9950 0000, $18.8bn), moves suppressed commodity codes to non-trade codes ($7.2bn), and estimates some trade only by chapter. BACI holds Flows only at HS6, so this value cannot appear in it. |
| Not attributed to a Market | 3.4 | Country suppressed for confidentiality (code "YY"), ships' stores and similar. |
| Precious metals (chapter 71) | 36.8 | Mostly non-monetary gold (7108): $65.2bn in HMRC against $33.1bn in BACI, almost all to Switzerland (+$28.2bn by Market). London's bullion trade is declared by the UK, but BACI reconciles it with partner reports that record it differently. |
| Everything else | 18.3 | About 5% of the remaining trade. Machinery (84, +$9.3bn), electrical goods (85, +$4.5bn) and organic chemicals (29, +$3.1bn) are higher in HMRC; inorganic chemicals (28, −$1.7bn), beverages (22, −$1.4bn) and ships (89, −$1.2bn) are higher in BACI. Plausible reasons are the CIF→FOB adjustment of partner reports, HMRC's monthly exchange rates against one annual average, and the HS 2022 → HS 2012 conversion, which moves value between Products. |

**At pair level the sources agree on the shape of UK trade.** 219,668 Product–Market pairs appear in both. Their values have a rank correlation of **0.89**, and 55% of BACI's value in those pairs lies within 25% of HMRC's figure. BACI also has 108,006 pairs that HMRC does not attribute at HS6 × Market level (most of them small: value HMRC suppressed, booked as low-value or never declared, recovered by BACI from the partner's import report), and HMRC has 1,739 that BACI lacks.

**Conclusion.** Every Exporter and Importer in the Flows maps to an ISO3 code with gravity features, and missing GDP and population follow one written rule with the source of each value recorded. Against HMRC, BACI understates UK goods exports by about a sixth in total, but for explained reasons: value HMRC does not attribute to a Product or Market, and gold. Outside those, the two differ by about 5% and rank pairs alike. So BACI is a sound target for the *shape* of UK trade, which is what the rankings use. Two cautions carry forward to the model card: chapter 71 (precious metals) is measured very differently by the two sources, and Export Gaps for small pairs inherit BACI's partner-report reconstruction.

## 3. Policy features: Trade Agreements and Trading Blocs

The policy features are the only columns a Scenario changes, so they must be right for every year. They come from the WTO's Regional Trade Agreements database (RTA-IS), "Export all RTAs" file. It lists every notified agreement with its goods entry-into-force date and **per-member entry and exit dates** (`docs/research/trade-agreement-coverage.md`). CEPII Gravity, the usual source in trade economics, stops in 2021, so it misses UK–Australia, UK–New Zealand and the UK's CPTPP accession.

For each pair and year, we build:

- `trade_agreement`: a goods Trade Agreement is in force between the two countries;
- `customs_union`: the agreement is a Customs Union (WTO type "CU");
- `eu_o/_d`, `cptpp_o/_d`, `efta_o/_d`: Trading Bloc membership of the Exporter and the Importer.

Each rule below follows how the WTO and Gravity code agreements:

- **The 1 July rule** (Gravity's): a membership counts in a year if it began before 1 July and had not ended by then. EU–Japan (1 Feb 2019) counts from 2019, and UK–Australia and UK–NZ (31 May 2023) from 2023. The UK's CPTPP accession (15 Dec 2024) counts only from 2025, so the 2024 Baseline does not include it.
- **A bloc that is a party is expanded to its members at each date.** "EU – United Kingdom" lists only "European Union" as the EU side. Croatia is still covered, because it is an EU member. An "EU – X" agreement links the EU side with X, not X's partners with each other.
- **The UK in 2020:** the UK left the EU (the Trading Bloc) on 31 Jan 2020. It stayed in the EU Customs Union and the EU's agreements until 31 Dec 2020 (the transition period). So for 2020 the UK's `eu_o` is 0, while its `customs_union` with EU members is still 1. The "EU Customs Union only" Scenario relies on this distinction.
- **Partial-scope agreements** (WTO type "PSA", such as the GSTP among developing countries) cut tariffs on short product lists only, so they do not count as Trade Agreements. Gravity's `fta_wto` leaves them out too.

This applies the course's **feature engineering** with explicit **data lineage**. Every flag traces back to a dated record in a primary source. `python -m exportspredictor.policy` rebuilds the tables from scratch; it is a step in `scripts/rebuild.py`.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world carries its own trade_agreement column (section 4).")
else:
    policy_files = policy.policy_paths()  # written by `python -m exportspredictor.policy`; read here, not rebuilt
    agreements = pl.read_parquet(policy_files["agreements"])
    trading_blocs = pl.read_parquet(policy_files["trading_blocs"])
    print(agreements.height, "pair-years with a Trade Agreement;", trading_blocs.height, "country-years in a tracked Trading Bloc")

The UK's policy features for a few Markets show the known dates:

- the EU Customs Union until 2020;
- the UK–EU agreement from 2021;
- Australia and New Zealand from 2023;
- no CPTPP membership yet in 2024.

In [ ]:
if not FIXTURE_MODE:
    uk_rows = pl.DataFrame(
        {"exporter": "GBR", "importer": ["FRA", "JPN", "AUS", "NZL", "MEX", "USA"]}
    ).join(pl.DataFrame({"year": list(range(2018, 2025))}, schema={"year": pl.Int32}), how="cross")
    uk_policy = policy.attach_policy(uk_rows, agreements, trading_blocs)
    display(uk_policy.pivot(on="year", index="importer", values="trade_agreement"))
    display(uk_policy.pivot(on="year", index="importer", values="customs_union"))
    display(uk_policy.filter(pl.col("importer") == "FRA").select("year", "eu_o", "cptpp_o", "efta_o"))

### Cross-check against CEPII Gravity, 2012–2021

Gravity V202211 codes `fta_wto` (per pair) and `eu_o/_d` (per country) up to 2021. It uses the same WTO database, with corrections by Thierry Mayer. We compare every pair-year that Gravity has. Most pairs have no agreement in either source, so the plain agreement rate flatters us. We therefore also report `positive_overlap`: of the pair-years that either source codes as in an agreement, the share that both do (the Jaccard index).

In [ ]:
if not FIXTURE_MODE:
    gravity_policy = policy.read_gravity_policy()  # from the gravity step's extract (section 2)
    crosscheck, disagreements = policy.gravity_crosscheck(agreements, trading_blocs, gravity_policy)
    display(crosscheck)

These are the results on the WTO export downloaded on 16 Nov 2026:

- **Trade Agreement:** the agreement rate is 95.5% over 549,900 pair-years, and the positive overlap is 70.6%. Both sources code 58,938 pair-years; 21,426 are ours only and 3,162 are Gravity only.
- **EU membership:** the agreement rate is 99.96%. The one difference is Croatia in 2013. Croatia joined on 1 July 2013, which is not *before* 1 July, so here it counts from 2014. Gravity counts any accession during the year.

Most of the **ours only** pair-years have a clear source:

- **Countries that Gravity never codes in any agreement (14%).** Gravity has no agreement at all for Montenegro, Bermuda or Yemen, and almost none for Liechtenstein. Yet the WTO lists, for example, EU–Montenegro (2008) and the Pan-Arab FTA.
- **Romania before 2018 (5%).** Gravity codes Romania's EU agreements only from 2018, although Romania joined the EU in 2007. This is a gap in Gravity.
- **The African Continental FTA (22%).** The WTO records it in force from 30 May 2019. Gravity does not code it, perhaps because trading under it began only in 2021.
- **The EU's Economic Partnership Agreements with African and Caribbean states (20%):** Eastern and Southern Africa, SADC, CARIFORUM and Ghana. They are notified to the WTO and applied, but Gravity's corrected `fta_wto` leaves most of these pairs out.
- **The rest (39%):** mostly COMESA accessions, SADC, EU–Serbia and EU–Central America. Here Gravity's corrections drop members that the export lists without dates.

Most of the **Gravity only** pair-years also have a clear source:

- **Territories missing from the WTO member lists (17%).** Mayotte, Réunion and other French overseas regions are part of the EU, but the WTO lists only France.
- **Haiti (17%).** Haiti signed the CARIFORUM agreement but has not ratified it. The WTO gives its entry date as 9999.
- **Latin American agreements notified as partial scope under LAIA**, such as Argentina–Bolivia and Argentina–Chile. We exclude these because they are PSAs.

The changes that the Scenarios and the Event Backtest rely on agree in both sources: EU–Japan from 2019, the UK's EU Customs Union to 2020, UK–EU and UK–Japan in 2021. Japan's only disagreements are CPTPP with Canada and New Zealand (in force 30 Dec 2018) and Japan–Mongolia (2016), which Gravity does not code.

UK pairs account for 368 of the disagreeing pair-years: 252 are ours only and 116 Gravity only. They are of the kinds above: Romania, Montenegro and Liechtenstein, the EPA partners, and a year's difference in when EU–Central America started. One more kind appears only for the UK. In 2021, Gravity keeps the UK in EU agreements that the UK did not roll over at the start of 2021, for example with Algeria, Syria, Andorra and the overseas territories. The WTO lists no UK agreement with these partners.

In [ ]:
if not FIXTURE_MODE:
    display(
        disagreements.filter(pl.col("feature") == "trade_agreement")
        .explode("years")
        .group_by("ours", "gravity")
        .agg(pair_years=pl.len(), countries=pl.concat_list("exporter", "importer").explode().n_unique())
    )
    display(disagreements.filter((pl.col("exporter") == "GBR") | (pl.col("importer") == "GBR")).head(20))

**Conclusion.** The policy features come from dated primary records, follow the same 1 July rule as CEPII Gravity, and agree with Gravity on every agreement the Scenarios and Event Backtests rely on. Where the two sources differ, the reasons are known and mostly gaps in Gravity. These columns are the only ones a Scenario changes (section 6), so their correctness bounds the meaning of every Scenario Effect.

## 4. The feature table

Every model input is computed in one place, `exportspredictor.features`, which the notebook (training) and the API (serving and Scenarios) both import, so a feature cannot drift between the two. This is the course's **feature engineering**, built under explicit **leakage** rules.

| Group | Features | What it captures |
|---|---|---|
| Gravity | log GDP and population of both countries, log distance, contiguity, common language, colonial tie, common colonizer | size and closeness of the two countries (section 2) |
| Supply | `log_supply_loo`: the Exporter's exports of the Product to all *other* Importers; `rca_loo`: its revealed comparative advantage in the Product | what the Exporter makes and sells well |
| Demand | `log_demand_loo`: the Importer's imports of the Product from all *other* Exporters; `high_income_share_loo`: the share of those from high-income Exporters | what the Market buys, and from whom ("Romania buys medicines from developed economies") |
| Product | `sector`: the HS2 chapter | Sector-level differences |
| Relatedness | `relatedness`: how close the Product is to the Products the Exporter already exports competitively, with proximity fitted on 2012–2021 | capabilities next to what the Exporter already does well |
| Re-export | `reexport_ratio`: the Importer's own exports of the Product against its imports of it | Re-export Hubs, whose imports serve other Markets (ADR 0005) |
| Policy | Trade Agreement, Customs Union, EU, CPTPP and EFTA membership of each side (section 3), and the Applied Tariff, `log_applied_tariff` = ln(1 + t) (ADR 0006) | the only columns a Scenario changes; monotone-constrained, +1 for memberships and −1 for the tariff (ADR 0002) |

Market import growth (the Importer's imports of the Product against three years earlier) is also built, but the ablation below dropped it, so the model does not train on it.

### Leakage rules

- **Leave-one-out supply and demand.** A row's own Flow is subtracted from every aggregate it feeds. For a UK row, UK supply of the Product excludes the target Market (ADR 0001). Otherwise the feature would contain the answer, and the out-of-fold model would copy Actual Exports back, shrinking every Export Gap. Revealed comparative advantage also leaves the row's Flow out of the Exporter's and the world's totals.
- **What leave-one-out does not remove: the UK inside other rows' demand.** Leave-one-out is per row, so a *non-UK* row's demand features (`log_demand_loo`, `high_income_share_loo`) leave out that row's Exporter but still include the UK's Flow to the same Market. France's whisky exports to Japan, for example, carry Japan's whisky imports from everyone but France, UK whisky included, and the fold model that holds out UK–Japan whisky still trains on that row. This means held-out UK Flows are not perfectly invisible: no fold model can read a held-out pair's Actual Exports from its own features or label, so Export Gaps are not shrunk by memorisation, but where the UK supplies a large share of a Market's imports of a Product, the non-UK rows teach the model a demand level that includes UK sales, which can pull that pair's Export Potential slightly towards its Actual Exports. ADR 0001 records this as an accepted consequence: closing it would mean rebuilding every non-UK row's demand features without the UK, once per fold.
- **No features from the pair's trade in other Products.** Total UK–Japan trade, for example, is left out on purpose. It is high largely *because* of the policy conditions between the two countries, so it would absorb the policy effect. A Scenario that flips only the policy columns would then understate the Scenario Effect.
- **Training years only.** The clip ranges and the Sector encoder (`FeatureTransform`) are fitted on training years. Test-year data cannot reach them, and serving inputs are clipped to the range seen in training (as tariffs would be, ADR 0002). Aggregates are computed within each year, so a year's features never use a later year's Flows.

The test suite checks each rule (`tests/test_features.py`). It changes the UK–Japan whisky Flow and confirms that this row's supply and demand features stay the same, while the same Product's UK–Canada row changes. It moves the UK's other-Product exports from Japan to Canada and confirms that no UK–Japan whisky feature moves. And it distorts the test years' Flows a thousandfold and confirms that the fitted transform is unchanged.

### Training rows: non-zero Flows plus sampled relevant zeros

BACI records only non-zero Flows, but a model of Export Potential must also learn where trade does *not* happen. The full Exporter × Importer × Product × year grid has about 3.45 bn cells, too many to train on, and most of it is uninformative (Malta not exporting jet engines). So the training rows are:

- **every non-zero Flow**, with weight 1;
- **a random sample of relevant zero cells**, with weight 1/rate. A cell is a relevant zero when the Exporter exports the Product to someone and the Importer imports it from someone that year, but not from each other. These are the zeros that carry information: both sides are active in the Product, and yet no trade happens between them.

Within the top-100 Exporter scope (ADR 0004) there are about 59 M relevant Zero Cells a year against 10–11 M non-zero Flows. **Every UK Zero Cell is kept**, at weight 1 (about 0.53 M a year), because New Market and New Product Opportunities are themselves UK Zero Cells. Other Exporters' Zero Cells are kept at 5%, about 2.9 M a year, and each carries weight 20, so weighted totals stay unbiased: this is the Horvitz–Thompson estimator, which the fixture test checks against an exact count. This goes beyond the course's treatment of class imbalance. Instead of rebalancing classes, we subsample the majority and reweight it, so the Tweedie loss still sees zeros in their true proportion. Whether a cell is kept depends only on a hash of its keys and a seed, so the sample is reproducible and never shifts when other data change. The rate and seed are recorded in the artefact metadata. v1 kept 2% of every Exporter's Zero Cells, the UK's included; v2's 5% for other Exporters was settled by the learning curve below.

### The UK grid for scoring

For scoring, the UK's grid of every Product × Market is built in full for each year, with no sampling: about 5,200 Products × 225 Markets ≈ 1.17 M pairs. About 72% of them have no Actual Exports. Each pair's Export Potential comes from its out-of-fold model.

On real data, `features.build()` writes one file per year for the training rows and one for the UK grid (`derived/features_V202601/`). It builds 2012–2024 in about 15 minutes, with a peak memory of about 14 GB, inside 32 GB, because each year is built from that year's Flows and, for Market import growth, those of three years earlier. Proximity for Relatedness is fitted once on 2012–2021, before the validation year.

In [ ]:
from exportspredictor import features

if FIXTURE_MODE:
    feature_world = build_world()
    feature_table = features.build_features(
        features.training_rows(feature_world.flows, rate=0.5), feature_world.flows, feature_world.sources
    ).lazy()
else:
    feature_dir = features.features_dir()  # written by `python -m exportspredictor.features`; read here, not rebuilt
    feature_table = pl.scan_parquet(feature_dir / "training_*.parquet")

is_zero = pl.col("value_usd") == 0
feature_table.group_by("year").agg(
    rows=pl.len(),
    sampled_zeros=is_zero.sum(),
    relevant_zeros_estimated=pl.col("weight").filter(is_zero).sum(),
).sort("year").collect()

On real data, 2023 has 10.9 M non-zero Flows from the top-100 Exporters and 3.46 M Zero Cells (0.53 M of them the UK's, all kept), 14.3 M rows in all. The weights estimate about 59 M relevant Zero Cells. Across 2012–2024 there are about 176 M training rows (163 M in the 2012–2023 years the fold models train on), about 5.5 GB of Parquet.

In [ ]:
feature_table.filter(
    (pl.col("exporter") == "GBR") & (pl.col("importer") == "JPN") & (pl.col("hs6").cast(pl.String) == "220830")
).sort("year").tail(1).collect().transpose(include_header=True, column_names=["value"])

### Which new features earn their place: the ablation

v2 adds four candidate features: the Applied Tariff (ADR 0006), Relatedness, Market import growth and the re-export ratio. More inputs are not free: a feature that carries no signal still offers the trees spurious splits, which is variance without any fall in bias (course: *overfitting, underfitting and the bias–variance trade-off*). So each one must earn its place in a **one-at-a-time ablation** (drop-column importance, the retrain-without-it cousin of the course's *permutation importance*):

- one model with every feature, then one model without each new feature in turn;
- v1's configuration (depth 8, Tweedie variance power 1.3, early stopping on 2022), so that only the inputs change;
- one fold of UK pairs held out (ADR 0001) and the Sector subset `training.SECTOR_SUBSET` (fish, beverages, pharmaceuticals, plastics, knitted apparel, electrical machinery, vehicles), to keep each model to a few minutes on the GPU;
- trained on 2012–2021 and measured on **2022 only**: the weighted Tweedie deviance of the validation rows (other Exporters and the held-out UK pairs), and the within-Product Spearman rank correlation of the held-out UK pairs, the ranking Advisors see. **2023, the test year, is never read** (`experiments.TestYearRead`, and a test that watches every file the runs open).

A feature **hurts** when the model without it is better on *both* measures; it is then dropped. The Applied Tariff is always kept, because tariff Scenarios need it. Every model is scored on the same held-out rows, as the course asks when comparing alternatives.

In [ ]:
from exportspredictor import experiments

if FIXTURE_MODE:
    experiment_results = experiments.run(experiments.fixture_data(build_world()), experiments.FIXTURE_PLAN)
else:
    # Saved by `python -m exportspredictor.experiments build | ablation | curve` (about an hour on the GPU).
    experiment_results = experiments.load()

ablation = pl.DataFrame(experiment_results["ablation"])
print("Dropped:", [f for f in features.ALL_FEATURES if f not in experiment_results["kept_features"]] or "none")
ablation.select(
    "dropped", "features", "training_rows", "rounds", "validation_deviance", "deviance_change",
    "uk_spearman", "spearman_change", "hurts",
)

On the real data (runs of 17 Nov 2026 on the RTX 3080, about 2–4 minutes a model; fold 0, the Sector subset, 17.7 M training rows, 3.2 M validation rows and 36,525 held-out UK pairs, all of 2022):

| Model | 2022 deviance | Change | UK Spearman | Change | Verdict |
|---|---|---|---|---|---|
| every feature | 416.2 | | 0.600 | | |
| without Relatedness | 426.4 | +10.2 | 0.609 | +0.009 | kept: deviance worse without it |
| without Market import growth | 412.0 | −4.1 | 0.612 | +0.012 | **dropped**: better on both without it |
| without the re-export ratio | 418.7 | +2.5 | 0.614 | +0.014 | kept: deviance worse without it |
| without the Applied Tariff | 416.4 | +0.2 | 0.605 | +0.005 | kept by rule (Scenarios need it) |

So `market_import_growth` leaves the model inputs (`features.ABLATED_FEATURES`; it is still built, so the leakage tests keep guarding it), and the model trains on 25 features. Two cautions keep this honest. First, *every* model without one feature ranks the held-out UK pairs slightly better (+0.005 to +0.014), which looks more like one-fold noise in the ranking measure than a real effect of each feature; the deviance changes are the more informative column, and only Relatedness moves it by more than 1%. Second, the Applied Tariff barely moves either measure: the model associates little with it beyond what the Trade Agreement and Trading Bloc flags already carry (a likely reason is that tariffs are zero under most agreements, so the two move together). It stays because tariff Scenarios need it, under its −1 monotone constraint. Most models stop at v1's 1,500-round cap, so the hyperparameter search (#54) raises the cap to 5,000.

### How many rows: the learning curve

The real training rows do not fit in the GPU's memory, so other Exporters' rows are sampled (ADR 0004): every UK row is kept, other Exporters' Zero Cells are kept at a **zero rate**, and their rows are then kept at a **non-UK rate**, each sampled row weighted by the inverse of its rate. A **learning curve** asks whether those rates cost accuracy. It trains the same model on the kept features at non-UK rates of 0.1, 0.3 and 0.65 and zero rates of 2%, 5% and 10%, on the same fold, Sectors and 2022 validation rows as the ablation.

This is the course's bias–variance trade-off read along the data axis rather than the complexity axis. If the curve is still falling at the highest rate, the model is variance-limited and more rows would help; if it has flattened, the remaining error is bias, which more data cannot remove (course: *more data narrows uncertainty but cannot add signal the features lack*). The rows are built once at a 10% zero rate and cut to lower rates by the same hash rule (`features.subsample_zeros`), so the samples nest and one feature build serves every point. `full_data_non_uk_rate` is the highest non-UK rate at which the *full* training rows (every Sector, 2012–2021) still fit the VRAM budget at that zero rate: a point with `fits_budget` false is not reachable in the final build.

In [ ]:
curve = pl.DataFrame(experiment_results["learning_curve"])
curve.select(
    "zero_rate", "non_uk_rate", "training_rows", "rounds", "validation_deviance", "uk_spearman",
    "full_data_non_uk_rate", "fits_budget",
)

In [ ]:
pl.DataFrame(experiment_results["plateau"])

On the real data (same runs, the 25 kept features):

| Zero rate | non-UK 0.1 | non-UK 0.3 | non-UK 0.65 | UK Spearman at 0.1 / 0.3 / 0.65 | highest non-UK rate the full rows allow |
|---|---|---|---|---|---|
| 2% | 495.8 | 441.1 | 413.6 | 0.615 / 0.614 / 0.609 | 0.84 |
| 5% | 495.9 | 438.6 | **412.1** | 0.614 / 0.608 / 0.607 | 0.73 |
| 10% | 456.0 | 429.6 | 413.8 | 0.598 / 0.601 / 0.600 | 0.59 (0.65 does not fit) |

**The curve has not plateaued in deviance.** Going from a non-UK rate of 0.3 to 0.65 still lowers the 2022 deviance by 6% at zero rates of 2% and 5%, and by 4% at 10%. On the course's reading the model is still variance-limited on the large Flows that dominate the deviance, so it should take every non-UK row the VRAM budget allows: `pipeline.REAL_NON_UK_RATE` stays `None` (derived from the budget, about 0.73 at 5% zeros for 2012–2021 rows, a little less for the final 2012–2023 rows). The UK ranking measure, by contrast, is flat: it moves by less than 0.01 across the whole grid, so more rows sharpen the scale of Export Potential more than its order within a Product.

**The zero rate stays at 5%** (`features.ZERO_SAMPLING_RATE`). At the highest non-UK rate it has the lowest deviance (412.1, against 413.6 at 2% and 413.8 at 10%), its UK Spearman is within 0.003 of 2%, and it leaves room for a non-UK rate of 0.73. 10% helps most at small non-UK rates, but it costs UK Spearman (about 0.60 against 0.61) and pushes the budget's rate down to 0.59. The differences between zero rates are small next to the effect of the non-UK rate. ADR 0004's last resort (GPU external memory) stays open: a curve still falling at the budget is the evidence that would justify it, once the model is tuned.

The package settings that follow from these tables are `features.FEATURES` (the kept features), `features.ZERO_SAMPLING_RATE` (the zero rate the real feature files are built at) and `pipeline.REAL_NON_UK_RATE` (`None`: the highest non-UK rate that fits the VRAM budget). Both tables travel with every artefact, in its metadata under `experiments`, beside the rates the artefact was trained with:

In [ ]:
{
    "kept_features": experiment_results["kept_features"] == features.FEATURES,
    "zero_sampling_rate": features.ZERO_SAMPLING_RATE,
    "non_uk_rate": pipeline.REAL_NON_UK_RATE or "derived from the VRAM budget",
}

**Conclusion.** One package function builds every model input for training and serving alike, which removes training/serving skew by construction (course: *training/serving skew and feature stores*). The leakage rules are enforced by tests rather than by care: leave-one-out supply and demand stop a row from seeing its own Flow, and leaving out the pair's trade in other Products keeps the policy effect in the policy columns. Sampled, reweighted zeros let a model of 165 M rows learn where trade does not happen without building the 3.45 bn-cell grid. The ablation and the learning curve, both measured on 2022 alone with the test year untouched, kept Relatedness, the re-export ratio and the Applied Tariff and dropped Market import growth; and they showed the deviance still falling at the highest non-UK rate, so the model takes every non-UK row the VRAM budget allows, with 5% of other Exporters' Zero Cells.

## 5. Training: tuning, out-of-fold fold models and the Baseline

The model is a single gradient-boosted tree ensemble (XGBoost, ADR 0002), trained on the GPU (`device="cuda"`). It extends the course's tree-ensemble material in four ways:

- **Tweedie loss.** Flows are non-negative, often zero and extremely skewed, so squared error on raw values would chase a few huge Flows. The Tweedie objective models a log-mean with a variance that grows with the mean, which suits exactly that (course: skewed targets and loss choice).
- **Monotone constraints** (+1 on every policy feature, ADR 0002): adding a Trade Agreement, Customs Union or Trading Bloc membership never lowers Export Potential. The model *associates* an agreement with higher Flows; it does not show cause.
- **Out-of-fold cross-fitting for the UK** (ADR 0001, beyond the course's k-fold cross-validation). UK Product–Market pairs are split into 5 folds by a hash of the pair, so every year of a pair shares a fold. Fold model *k* trains on all non-UK rows plus the UK pairs outside fold *k*, and scores only fold *k*. A pair's Export Potential therefore never comes from a model that saw its Actual Exports, so the Export Gap is not shrunk by memorisation. `tests/test_folds.py` checks this by distorting one fold's UK labels a thousandfold: that fold's scores do not move at all, while every other fold's do.
- **Training data that does not fit in memory.** The 162 M training rows of 2012–2023 (10.2 M UK rows, Zero Cells included, and 152 M other rows) are about 16 GB as floats. They are streamed to XGBoost one year at a time through a data iterator into a `QuantileDMatrix`, which stores only the binned features. Even binned, all of them would overflow the RTX 3080's 10 GB once training starts, so the real run keeps the largest hash sample of *non-UK* rows that the VRAM row budget allows (ADR 0004): 58.9%, reweighted by 1/0.589 (the same Horvitz–Thompson idea as the sampled zeros in section 4), about 100 M rows in all: the budget measured on this card by `scripts/gpu_check.py` is 100,032,198 rows, about 9% below the about 110 M rows the spec planned around. Every UK row is kept. Training peaked at about 7.5 GB of the card's memory.

### The time split and the search

The years are split as the course recommends for data with a time order (course: train/validation/test splits and leakage):

| Role | Years | Used for |
|---|---|---|
| training (search) | 2012–2021 | fit each candidate configuration |
| validation | 2022 | early stopping and choosing the configuration |
| test | 2023 | the evaluation (section 7); never read here |
| refit | 2012–2023 | the final fold models that score the Baseline |
| Gap Period | 2022–2024 | each year scored by the refit fold models; the Export Gap is the mean of the three yearly gaps |
| Baseline | 2024 (provisional) | the starting point of every Scenario and Blocked Opportunity value |

The search replaces v1's five hand-picked configurations. It goes beyond the course's grid and random search to **Bayesian optimisation**: Optuna's Tree-structured Parzen Estimator (TPE) models which regions of the space gave low validation loss so far and samples the next configuration there. A **Median pruner** stops a trial early when, round by round, its validation loss is worse than the median of earlier trials at the same round, so poor configurations cost minutes rather than the full budget. v1's winning configuration is enqueued as trial 0, so the search can only match or beat it on its own objective.

| Hyperparameter | Range |
|---|---|
| `max_depth` | 4–12 |
| `min_child_weight` | 1–1000, log scale |
| `subsample` | 0.5–1 |
| `colsample_bytree` | 0.4–1 |
| `reg_lambda` (L2 penalty on leaf weights) | 0.1–100, log scale |
| `tweedie_variance_power` | 1.1–1.9 |

`eta` is fixed at 0.1, with a cap of 5,000 rounds and early stopping after 50 rounds without improvement on 2022 (course: early stopping as regularisation); trials that hit the cap are logged. The search runs in two stages, both training on 2012–2021 at the final sampling rates and validating on 2022 with one fold of UK pairs held out:

1. **TPE on the Sector subset** (fish, beverages, pharmaceuticals, plastics, knitted apparel, electrical machinery and vehicles), for a budget of about two hours on the GPU.
2. **The best three configurations retrained on every Product**; the lowest deviance wins.

Candidates with different variance powers optimise different losses, so they are all compared on one fixed objective, the weighted Tweedie deviance at power 1.5 on the held-out 2022 rows. The within-Product Spearman of the held-out UK pairs is logged for every trial as a guardrail: a lower deviance bought with a worse ranking of UK Markets would be a bad trade for Advisors, so the notebook flags a winner whose Spearman is more than about 0.01 below v1's configuration's. The study is stored in SQLite under the data directory and resumes if the run is killed (`python -m exportspredictor.search`). The feature transform (clip ranges and Sector encoder) is fitted on the search's training years for the search, and on 2012–2023 for the refit, never on a year it is scored on.

**Round scaling (a heuristic).** The fold models reuse the tuned configuration with no nested tuning. They train on 2012–2023, about 1.2 times the rows of the search's 2012–2021, so the early-stopped round count is scaled by that ratio of training rows (`search.scaled_rounds`). More rows at the same learning rate usually support more rounds before overfitting, but nothing validates the scaled count: it is a rule of thumb, recorded in the metadata under `round_scaling`.

In [ ]:
if FIXTURE_MODE:
    world = build_world()
    bundle = pipeline.build_artefacts(
        world, source="fixture", experiment_results=experiment_results, parts=pipeline.Parts.every()
    )
else:
    # Trained and saved by `python -m exportspredictor.pipeline` (a step of scripts/rebuild.py, several
    # hours on an RTX 3080, evaluation included); the notebook reads the newest saved version. The Event
    # Backtests, the constraint comparison and the gain ladder are opt-in there (`pipeline.Parts`); a part
    # the run skipped is reported below as not run for this version.
    bundle = artefacts.load_latest(artefacts_dir())
print(reporting.provenance(bundle))
{key: bundle.metadata[key] for key in ("version", "source", "device", "params", "rounds", "tuning_seconds", "training_seconds", "non_uk_sampling_rate")}

The search's trial table (stage 1, the Sector subset), in trial order. Trial 0 is v1's configuration; a pruned trial has no deviance, and its rounds are those it ran before it was stopped:


In [ ]:
from exportspredictor import search

search_results = bundle.metadata["search"]
trials = pl.DataFrame(search_results["trials"]).unnest("params")
trials.select(
    "number", "state", "v1_config", *search.SPACE, "rounds", "hit_round_cap", "validation_deviance", "uk_spearman",
    "seconds",
)


Stage 2, the best configurations retrained on every Product, and the guardrail against v1's configuration (both Spearman values from stage 1, where v1's configuration was trained):


In [ ]:
display(
    pl.DataFrame(search_results["confirmation"]).select(
        "trial", "rounds", "hit_round_cap", "training_rows", "validation_deviance", "uk_spearman", "seconds"
    )
)
winner, v1 = search_results["winner"], search_results["v1"]
print(f"Winner: trial {winner['trial']}, {winner['params']}, {winner['rounds']} rounds")
print(
    f"UK Spearman (stage 1): winner {search_results['trials'][winner['trial']]['uk_spearman']}, "
    f"v1's configuration {v1['uk_spearman']}, change {search_results['spearman_vs_v1']}"
)
if search_results["loses_to_v1"]:
    print(f"FLAG: the winner ranks UK Markets worse than v1's configuration by more than {search.SPEARMAN_GUARDRAIL}.")
else:
    print(f"Guardrail passed: the winner's UK Spearman is within {search.SPEARMAN_GUARDRAIL} of v1's or above it.")
print("Trials that hit the round cap:", search_results["capped_trials"] or "none")
bundle.metadata["round_scaling"]


One point of the learning curve (section 4) again, with the tuned configuration instead of v1's: the same Sectors, fold and rates. If the tuned model gains a lot here, the curve's conclusion about rates was drawn with an under-fitted model.


In [ ]:
recheck = search_results["learning_curve_recheck"]
point = (recheck["non_uk_rate"], recheck["zero_rate"])
v1_point = next(row for row in experiment_results["learning_curve"] if (row["non_uk_rate"], row["zero_rate"]) == point)
pl.DataFrame(
    [
        {"configuration": name, **{k: row[k] for k in ("rounds", "hit_round_cap", "validation_deviance", "uk_spearman")}}
        for name, row in (("v1", v1_point), ("tuned", recheck["tuned"]))
    ]
)


On the real data (search of 17–18 Nov 2026 on the RTX 3080, `data/derived/search_V202601/`): stage 1 trained 27 trials in its two-hour budget on 16.8 M rows of the Sector subset (non-UK rate 0.6157, the largest the VRAM budget allows with the whole 2022 validation year beside the training rows; 5% of other Exporters' Zero Cells), validated on 3.2 M rows of 2022 and 36,525 held-out UK pairs. 20 trials finished and 7 were pruned at round 101, the first round the pruner may act. **No trial hit the 5,000-round cap**: v1's configuration stopped at 4,781 rounds, so v1's 1,500-round cap had been cutting it short.

| Stage 1 (Sector subset) | Depth | min_child_weight | subsample | colsample | reg_lambda | Variance power | Rounds | 2022 deviance | UK Spearman |
|---|---|---|---|---|---|---|---|---|---|
| trial 0: v1's configuration | 8 | 1 | 0.80 | 0.80 | 1.0 | 1.30 | 4,781 | 399.5 | 0.622 |
| trial 15 | 12 | 3.05 | 0.92 | 0.88 | 4.95 | 1.266 | 879 | **390.6** | 0.612 |
| trial 23 | 11 | 6.78 | 0.86 | 0.99 | 23.8 | 1.274 | 1,321 | 391.0 | 0.613 |
| trial 19 | 11 | 1.44 | 0.88 | 0.95 | 0.96 | 1.277 | 1,437 | 391.1 | 0.616 |

| Stage 2 (every Product, 84.1 M training rows, 13.6 M validation rows, 231,217 held-out UK pairs) | Rounds | 2022 deviance | UK Spearman |
|---|---|---|---|
| trial 15 | 1,198 | **364.3** | 0.528 |
| trial 23 | 1,688 | 365.6 | 0.531 |
| trial 19 | 1,829 | 365.8 | 0.536 |

**The winner is trial 15**: deeper trees (12) than v1's, slightly larger leaves, a stronger L2 penalty and a variance power of 1.27, close to v1's 1.3. It lowers the 2022 deviance by 2.2% against v1's configuration on the Sector subset, and it stops after a quarter of the rounds. The three confirmed configurations are within 0.4% of each other on every Product, so the search found a flat basin rather than a sharp optimum; deeper trees and a variance power near 1.25–1.3 are what they share.

**The Spearman guardrail is passed, narrowly.** On the Sector subset the winner's UK Spearman is 0.612 against v1's configuration's 0.622, a change of −0.0099, just inside the 0.01 tolerance. The lower deviance comes mostly from the large Flows that dominate it, while the ranking of UK Markets within a Product is a little worse; this is the trade-off the guardrail watches. Among the confirmed configurations trial 19 ranks best (0.536 on every Product), but its deviance is 0.4% higher, and the pre-registered objective is the deviance. The stage-2 Spearman values are lower than stage 1's because they cover every Product, including thinly traded ones, and are not comparable with the Sector subset's.

**The learning-curve point re-checked** (non-UK rate 0.65, zeros 5%, Sector subset): the tuned configuration reaches a deviance of 388.6 in 944 rounds, against 412.1 for v1's configuration at v1's 1,500-round cap, with the same UK Spearman (0.608 against 0.607). Tuning moves the whole curve down by about 6%, as much as going from a non-UK rate of 0.3 to 0.65 did, so #53's choice of rates was not an artefact of an under-fitted model.

The fold models train on 2012–2023 with the winner's 1,198 rounds scaled by the ratio of training rows (`round_scaling` in the metadata); the evaluation section tests the result on the untouched 2023.


### The Baseline table and the Gap Period

The refit fold models score the UK grid of each **Gap Period** year, 2022, 2023 and 2024, each pair by the fold model that held it out (ADR 0001), each year with that year's features. The grid is the union of the three years' grids, so a pair is scored in every year even if its Product was not traded in one of them, and a year with no Flow counts as $0 of Actual Exports. The **Export Gap** is the plain mean of the three yearly gaps (Export Potential minus Actual Exports), so gaps still add up across pairs. The Baseline table holds the three Gap Period means and, separately, the 2024 Export Potential: Scenarios and Blocked Opportunity values start from 2024 alone.

Why average? One year's Actual Exports are noisy: a single large shipment, or a re-routed contract, can move a pair's Flow several-fold from one year to the next, and the 2024 figures are provisional. Averaging three yearly gaps cuts that year-to-year variance, the same idea as bagging averaging many noisy trees (course: *ensembles: bagging averages to cut variance*), while keeping the gap recent. Why 2022–2024 and not five years? **2020 and 2021 are left out on purpose**: COVID-19 collapsed and then rebounded trade, and the end of the Brexit transition period (January 2021) reset UK–EU trade, so those years describe shocks rather than the UK's current fundamentals. Using them would bias the gaps of exactly the Markets Advisors care most about.

The artefacts also store each pair's 2024 model inputs and fold for the Scenarios and explanations to come, and a yearly history: Export Potential, Actual Exports and Export Gap for each Gap Period year, and the UK's Actual Exports for 2019–2021.

In [ ]:
baseline = bundle.baseline.join(bundle.products, on="hs6", how="left").join(
    bundle.markets.rename({"name": "market"}), on="iso3", how="left"
)
baseline.select(
    pairs=pl.len(),
    products=pl.col("hs6").n_unique(),
    markets=pl.col("iso3").n_unique(),
    pairs_with_exports=(pl.col("actual_exports_usd") > 0).sum(),
    actual_exports_usd=pl.col("actual_exports_usd").sum(),
    export_potential_usd=pl.col("export_potential_usd").sum(),
    baseline_export_potential_usd=pl.col("baseline_export_potential_usd").sum(),
    opportunities=(pl.col("export_gap_usd") > 0).sum(),
)

How one pair's Export Gap is formed: whisky (`220830`) to Japan, year by year. The mean of the three yearly gaps is the Export Gap in the Baseline table.

In [ ]:
whisky_japan = (pl.col("hs6") == "220830") & (pl.col("iso3") == "JPN")
display(bundle.history.filter(whisky_japan, pl.col("export_gap_usd").is_not_null()))
bundle.baseline.filter(whisky_japan).select(
    "export_potential_usd", "actual_exports_usd", "export_gap_usd", "baseline_export_potential_usd"
)

### Sanity check: the top Opportunities

A model can score well and still rank nonsense, so we read the largest Export Gaps and ask whether a trade economist would find them plausible. Two checks come first. No Excluded Product may appear at all (ADR 0005: they are dropped before scoring). Flagged Products and Re-export Hubs do stay in the rankings, with a caveat, so we measure how much of the top of the list they hold: if they dominated it, an Advisor would mostly be reading about gold bars and transit trade.

In [ ]:
from exportspredictor.exclusions import is_excluded

top_opportunities = baseline.filter(pl.col("export_gap_usd") > 0).sort("export_gap_usd", descending=True)
top_opportunities.select(
    "hs6", "description", "market", "opportunity_type", "export_potential_usd", "actual_exports_usd",
    "export_gap_usd", "flagged_product", "reexport_hub", "uncertain_opportunity",
).head(20)

In [ ]:
pl.concat(
    [
        top_opportunities.head(top).select(
            top=pl.lit(top),
            excluded_products=is_excluded().sum(),
            flagged_product_share=pl.col("flagged_product").mean(),
            reexport_hub_share=pl.col("reexport_hub").mean(),
            flagged_or_hub_share=(pl.col("flagged_product") | pl.col("reexport_hub")).mean(),
            uncertain_share=pl.col("uncertain_opportunity").mean(),
        )
        for top in (20, 100, 1000)
    ]
)

One Product an Advisor would know well, whisky (`220830`): the Markets with the largest Export Gaps, and the ones where the UK already exceeds its Export Potential.

In [ ]:
whisky = baseline.filter(pl.col("hs6") == "220830").sort("export_gap_usd", descending=True)
whisky_columns = ["market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"]
display(whisky.select(whisky_columns).head(8))
whisky.select(whisky_columns).tail(5)

On the real data (artefact version `20261118T042154Z`), the Baseline covers 1,156,050 UK Product × Market pairs, the union of the 2022, 2023 and 2024 grids without the Excluded Products (5,138 Products × 225 Markets). Of these, 416,566 had UK exports in at least one Gap Period year. Their Gap Period mean Actual Exports total $346 bn and Export Potential $315 bn; the 2024 Baseline Export Potential that Scenarios start from totals $367 bn. Read the results with these points in mind:

- **91% of pairs have a positive Export Gap**, but the gaps are small almost everywhere. This is what a conditional mean does: a pair with no exports but plausible fundamentals gets a small positive Export Potential, and the 734,084 *New Market* pairs hold only $9 bn of positive gap between them, against $112 bn for the 416,566 *Expand* pairs. So individual small gaps are noise, and only the *size* ranking is informative. On pairs with exports, the rank correlation between Export Potential and Actual Exports is 0.74, so the model reproduces the shape of UK trade without copying it (the pairs were held out).
- **The sanity checks pass.** No Excluded Product appears anywhere in the Baseline, so none reaches the top. Flagged Products hold 5% of the top 20, 4% of the top 100 and 1.4% of the top 1,000 (gold jewellery to Ireland and the USA). Re-export Hubs hold more: 15%, 25% and 29% (Ireland, Hong Kong, Belgium, the Netherlands). That is expected, since hubs are large importers of exactly the Products the UK makes, but it is why their rows carry a caveat that some of what they import moves on.
- **The top Opportunities are plausible.** All of the top 20 are *Expand* pairs in the UK's known strengths: aero-engines and their parts (France, Türkiye, Hong Kong, Germany, China, Taiwan, Korea), packaged medicines (France, China, Japan, Italy), diesel engines to the USA and France, and aircraft and under-carriages to the USA. These are large Markets for Products the UK makes well, where comparable Exporters sell more. None of the top 1,000 is an Uncertain Opportunity: these gaps are larger than the ordinary scatter of comparable pairs (section 7).
- **Yearly Export Potential can swing.** For the largest Opportunity, turbo-jet parts to France, the yearly Export Potential rises from $0.58 bn in 2022 to $1.09 bn in 2023 and $2.47 bn in 2024, while Actual Exports stay near $0.1 bn. The provisional 2024 data and France's very high re-export ratio for these parts (section 8) move it. Averaging the three years damps such swings, and the history on the Why? page shows them.
- **Some top gaps have explanations the model cannot see.** For whisky, Malaysia and the Netherlands lead, then Latvia and Ireland, which are hubs and transit Markets. **Saudi Arabia** ranks seventh, a Market where alcohol is banned, from a 2022–2023 Export Potential of about $50 M that falls to under $1 M in 2024. The **Russian Federation** is eighth: UK whisky exports there fell from $211 M in 2021 to $19 M in 2022 under the sanctions. No feature records sanctions or prohibition, so an Advisor would read these rows as Barriers the model does not know about, not as Opportunities.
- **Whisky's negative gaps are where the UK already over-performs**: Taiwan, France, the USA, Singapore and India, its long-established whisky Markets.

As always, these are associations: the model says that comparable Exporters sell more of these Products to these Markets, not that the UK would if it tried.

**Conclusion.** The TPE search on 2022 chose a depth-12 model with Tweedie variance power 1.27 and 1,198 rounds. Scaled to 1,453 rounds, the five out-of-fold fold models trained in 75 minutes on the RTX 3080, on 10.2 M UK rows and 59% of the 152 M other rows (the most the VRAM row budget allows). Every UK pair in each Gap Period year is scored by a model that never saw it. The Export Gap is the 2022–2024 mean, which steadies the ranking against one-off shipments while leaving out the COVID and Brexit-transition years. The top Opportunities are plausible UK strengths in large Markets, with no Excluded Product and few Flagged Products among them; the exceptions (sanctions, prohibition, hubs) are things no feature records. Two training compromises go into the model card: the 59% sample of other Exporters' rows, forced by GPU memory, and round scaling, a heuristic nothing validates.

## 6. Scenarios and Blocked Opportunities

A Scenario asks the trained model a what-if question: copy the 2024 Baseline rows of the affected Markets, change **only the policy columns**, and re-predict each pair with the same fold model that scored its Baseline (ADR 0001). The Scenario Effect is the difference. `exportspredictor.scenarios` does this for the notebook and the API alike, and checks on every run that no non-policy column moved (a leakage guard: a Scenario that also shifted, say, demand would report the demand change as a policy effect).

This extends two course concepts:

- **Ensembles: gradient boosting.** The fold models carry a +1 monotone constraint on every policy feature (ADR 0002), so switching a Trade Agreement on can never lower a prediction. The guarantee holds along observed values only: XGBoost sends a missing value down each split's learned default branch, which the constraint does not cover, so an added agreement sets a *known* Applied Tariff to 0 and leaves a missing one missing (ADR 0006). Zeroing missing tariffs had given 22,135 UK pairs a slightly negative Blocked Opportunity value in the first v2 artefact. A property-based test checks this over the fixture rows with random policy settings. Without the constraint, a tree could learn a spurious dip and a Scenario could show a negative effect from adding an agreement.
- **Correlation, multicollinearity and causation.** Flipping one feature and re-predicting is the same move as the course's counterfactual fairness test, but the answer is still learned from correlations in past trade. So we say "the model associates the CPTPP with +X%", never "joining CPTPP causes +X%". The Event Backtests of section 7 test these effects against real agreements.

The four Trading Bloc presets (wayfinder #8) change only the rows of the bloc's member Markets in the Baseline year:

| Preset | Policy columns set on member Markets' rows |
|---|---|
| EU re-join | `trade_agreement`, `customs_union`, `eu_o` (as for any pair of EU members in training) |
| EU Customs Union only | `customs_union` |
| CPTPP | `trade_agreement`, `cptpp_o`, with accession treated as complete |
| Leave | for each bloc the UK is in: its `<bloc>_o`, `trade_agreement` and `customs_union` switched off |

Because a membership counts in a year only if it began before 1 July, the 2024 Baseline has no UK CPTPP membership (accession took effect from December 2024), so the CPTPP preset is a live prediction that 2025 data can check. In 2024 the UK is in no tracked Trading Bloc, so Leave changes nothing, and the API says so.

In [ ]:
from exportspredictor import scenarios

preset_results = {
    name: scenarios.run(bundle.baseline_features, bundle.baseline, bundle.fold_models, scenarios.Scenario(preset=name), form=bundle.model_form)
    for name in scenarios.PRESETS
}
pl.DataFrame(
    [
        {
            "preset": scenarios.PRESETS[name].title,
            "pairs_changed": result.pairs.height,
            **result.total(),
            "blocked_opportunities": result.blocked_opportunities().height,
        }
        for name, result in preset_results.items()
    ]
)

On the real artefact (version `20261118T100524Z`), on CPU, from the 2024 Baseline:

| Preset | Pairs changed | Scenario Effect | Share of Baseline | Blocked Opportunities (≥20%, ≥$1M) |
|---|---|---|---|---|
| EU re-join | 138,726 (27 Markets) | +$75.8bn | +42.0% | 10,666 |
| EU Customs Union only | 138,726 | +$46.9bn | +26.0% | 5,984 |
| CPTPP | 56,518 (11 Markets) | +$7.8bn | +19.0% | 766 |
| Leave | 0 | 0 | | 0 |

Each preset now also moves the Applied Tariff (ADR 0006): joining sets a known tariff to 0 where no agreement existed (a missing one stays missing), and removing an agreement restores the Importer's 2021 MFN rate. The EU figures are large because the model associates intra-EU membership with much denser trade than a free-trade agreement alone; they describe what the model has learned from EU members' trade, not what re-joining would bring. The CPTPP effect is smaller in dollars: the UK already has agreements with 9 of the 11 members, so for those nine only the CPTPP flag changes, while Brunei and Malaysia also gain an agreement and a zero tariff. The CPTPP effect per Market:

In [ ]:
preset_results["cptpp"].by_market().join(bundle.markets.select("iso3", "name"), on="iso3", how="left")

### Precomputed Blocked Opportunity values

The rankings can also be ordered by **Blocked Opportunity value**: for every pair with no Trade Agreement in 2024, the Scenario Effect of adding one, re-predicted by the pair's fold model when the artefacts are built (`scenarios.blocked_opportunity_values`). It is 0 where an agreement already exists. A pair is a Blocked Opportunity when that effect is at least 20% of its Baseline Export Potential and at least $1M. The largest values:

In [ ]:
baseline.with_columns(
    blocked_opportunity=scenarios.is_blocked("blocked_opportunity_value_usd", "baseline_export_potential_usd")
).sort("blocked_opportunity_value_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "blocked_opportunity_value_usd", "blocked_opportunity"
).head(10)

### What the monotone constraints change

The +1 monotone constraint on every policy feature (ADR 0002) is a design choice, so we measure what it does rather than only state it. `constraints.compare` trains two models that differ *only* in the constraint, with the same rows, tuned parameters, round count and seed, and asks both the same Scenario questions for the same UK pairs: add a Trade Agreement where there is none, and remove it (with any Customs Union) where there is one. It counts the pairs that move the "wrong" way, an agreement that lowers Export Potential or a removal that raises it, and compares the size of the effects. This extends the course's gradient-boosting material: a monotone constraint is a form of domain knowledge built into the trees, a kind of regularisation that trades a little fit for answers that respect a known direction.

Both models follow ADR 0001: they train on all non-UK rows and the UK pairs outside one fold, and score only that fold's pairs, so no effect comes from a model that saw the pair. To keep the real run cheap (two models rather than ten fold-sized ones), the real comparison trains on three Sectors (22 beverages, 30 pharmaceuticals, 87 vehicles) and one held-out fold, with the tuned parameters and round count. The fixture run uses every Sector and fold.

In [ ]:
from exportspredictor.evaluation import NOT_RUN

monotone = bundle.evaluation.get("monotone_constraints")
print(f"Monotone-constraint comparison: {NOT_RUN}." if monotone is None else monotone["text"])
None if monotone is None else pl.DataFrame(
    [{"model": m["model"], "change": change, **m[change]} for m in monotone["models"] for change in ("add", "remove")]
)

On the real data (artefact version `20261118T042154Z`; 491 s on the RTX 3080 for both models, 1,453 rounds each), over the held-out fold's UK pairs in the three Sectors:

| Model | Adding a Trade Agreement (3,327 pairs) | Removing one (2,317 pairs) | Total effect of adding |
|---|---|---|---|
| Constrained | 0 pairs lower; median **+19.8%** | 0 pairs higher; median −17.7% | **+26.6%** |
| Unconstrained | **1,303 pairs (39%) lower**, down to −79%; median +8.0% | **757 pairs (33%) higher**, up to +191%; median −10.7% | +4.0% |

The two models' Baselines rank these pairs almost identically (Spearman 0.965), so the constraint costs little fit. Without it, almost two in five Blocked Opportunity questions would get an answer an Advisor could not defend: a Trade Agreement associated with *less* trade, sometimes by four-fifths. That is a spurious dip the trees learned from noise in past agreements, and it is what the constraint removes. The unconstrained share of wrong-signed answers is higher than v1's 26%, which fits a deeper model (depth 12 against 8) fitting more noise.

The comparison also shows the price, and we state it plainly. With the negative answers removed, the constrained model's total effect of adding an agreement is larger (+27% against +4%). The constraint does not make Scenario Effects more accurate; it makes their *direction* consistent. Their *size* is checked separately by the Event Backtests in section 7, which find that the model overstates real agreements several-fold. So the constraint is right for the question an Advisor asks ("would an agreement help here?"), and the size of each Scenario Effect stays an upper-end association, not a forecast.

### A gravity benchmark for the size of the Trade Agreement Scenario Effect

The monotone constraint fixes the *direction* of a Scenario Effect. Its *size* is still only what the trees learned. So we check it against the method economists use to measure Trade Agreement effects: a structural gravity regression fitted by Poisson pseudo-maximum likelihood (PPML; decision #33, `exportspredictor.ppml`). It is a benchmark only. It is never served, and it never competes on Export Potential, because its pair effects need the pair's own Flows, which a pair held out under ADR 0001 does not have.

**The regression.** Flows are summed to Sector level (HS2 chapters, non-Excluded Products, the model's top-100 Exporter scope, every Importer) for 2012, 2015, 2018, 2021 and 2024. Three-year intervals let trade adjust to an agreement, as the WTO/UNCTAD *Advanced Guide* recommends. The model is

$$\text{Flow}_{e,i,s,t} = \exp\big(a_{e,s,t} + b_{i,s,t} + c_{e,i,s} + \beta\,\text{TradeAgreement}_{e,i,t}\big)\cdot\varepsilon,$$

with Exporter-year, Importer-year and pair effects, each within a Sector. It has about a million pair effects. They absorb everything fixed about a pair (distance, language, and the fact that countries that sign agreements already traded a lot), so $\beta$ is identified only by pairs whose agreement status changed, compared with themselves before and with other pairs in the same year. $e^{\beta}-1$ is the proportional change in Flows the regression *associates* with a Trade Agreement.

**How it links to the course.** This is the course's linear regression and coefficient reading (section *Supervised models*), taken three steps further:
- a log link and Poisson loss, so that zeros stay in and large Flows are not distorted by taking logs (Santos Silva & Tenreyro 2006);
- fixed effects as a very large one-hot encoding, swept out by alternating projections rather than built as columns;
- standard errors clustered by Exporter–Importer pair, because a pair's years are not independent.

The fit is the course's training loop in its Newton form: iteratively reweighted least squares, run until the deviance stops changing. The course warns that correlation is not causation. Pair effects are the econometric answer to that, and even with them the wording stays associational: agreements are not randomly assigned.

**What it is compared with.** The model's mean Trade Agreement Scenario Effect over every UK pair without an agreement comes from the saved Blocked Opportunity values (out-of-fold, ADR 0001). It adds an agreement and zeroes the tariff, as every Scenario does (ADR 0006), and is measured against the 2024 Baseline. It is shown weighted by Baseline Export Potential, which is comparable to PPML because PPML also weights pairs by their size, and as a plain mean per pair. The literature's RTA estimates are listed with their sources in `docs/research/rta-effects.md`.

In [ ]:
# Fitted once by `python -m exportspredictor.ppml` (or the pipeline's PPML benchmark part) and saved with the
# artefact; read here, not refitted. A version without it says so.
ppml_benchmark = bundle.ppml_benchmark
None if ppml_benchmark is None else {"level": ppml_benchmark["level"], "years": ppml_benchmark["years"], **ppml_benchmark["ppml"]}

In [ ]:
reporting.ppml_benchmark(bundle)

On the real data (BACI V202601; 5,015,327 rows after dropping 15,228 that carry no information, 20,034 Exporter–Importer pairs; 17 iterations, 29 minutes on the CPU; the model side from artefact version `20261118T100524Z`):

| Estimate | Coefficient | Associated change | 95% interval |
|---|---|---|---|
| **PPML benchmark** (3-way fixed effects, Sector level, 2012–2024) | 0.050 (s.e. 0.028) | **+5.1%** | −0.4% to +11.0% |
| **Model: mean Trade Agreement Scenario Effect**, Baseline-weighted (678,216 UK pairs without an agreement) | 0.453 | **+57.3%** | |
| Model: the same, mean per pair (median +46.2%) | 0.563 | +75.5% | |
| Literature: Yotov et al. 2016, PPML without pair effects | 0.191 | +21% | |
| Literature: Head & Mayer 2014, structural gravity, median / mean | 0.28 / 0.36 | +32% / +43% | |
| Literature: Head & Mayer 2014, all gravity, median | 0.47 | +60% | |
| Literature: Yotov et al. 2016, PPML with pair effects and intra-national Flows | 0.557 | +75% | |
| Literature: Baier & Bergstrand 2007, "about doubles after 10 years" | ≈0.69 | ≈+100% | |
| **Literature range** | | | **+21% to +100%** |

Three readings follow, all associational.

- **The benchmark is small.** Over 2012–2024, a pair that comes under a Trade Agreement trades about 5% more within a Sector than the same pair did before, beyond what its Exporter's and Importer's Sector-year conditions explain. The interval only just includes zero. The coefficient is identified by 2,040 country pairs whose status changed (1,995 gained an agreement, 41 lost one), about 7% of the panel's value; the largest by value are Australia–China, Singapore–Hong Kong, Taiwan–Singapore and EU–Japan. It sits below the whole literature range, for known reasons: BACI has no intra-national Flows, which in the *Advanced Guide* more than double the estimate (0.191 to 0.409); the dummy is contemporaneous and observed every 3 years, while the Guide's phased-in total is about 1.0; and an agreement signed between partners that already trade on low tariffs may change little.
- **The model's effect is about eleven times the benchmark** (in percent; nine times in log points). On a Baseline-weighted basis, the model associates adding a Trade Agreement, with its tariff cut, with +57% UK Export Potential; the per-pair mean is +76%. This lies inside the literature range, near the median of all gravity estimates (+60%), but far outside the benchmark's interval on the same data and years. It agrees with the Event Backtests in section 7, where the model's EU–Japan effect (+35% to +42%) was many times the observed change.
- **Why the model can sit so high.** The trees learn the Trade Agreement association *across* pairs: pairs with agreements trade more, and the remaining features cannot absorb everything that makes those pairs close. The benchmark compares each pair with itself through its pair effect. The model cannot use pair effects, because a held-out UK pair has none (ADR 0001). So part of what the model attributes to an agreement is the closeness of the countries that sign them, the endogeneity Baier & Bergstrand warn about.

So the gravity benchmark confirms the direction of the Scenario Effect and supports reading its size as an upper end. An Advisor should treat "+57%" as "the model associates agreements with substantially more trade, plausibly in the range the literature reports", and never as a forecast of what one agreement will add.

**Conclusion.** Scenarios change only policy columns, including the Applied Tariff each agreement now carries with it, and re-predict each pair with its own fold model. The monotone constraint guarantees that an agreement never lowers Export Potential, and the comparison with an unconstrained model shows what that guarantee removes (39% wrong-signed answers). The bloc presets give large associations: EU re-join +42% of the bloc's 2024 Baseline, CPTPP +19%. A 3-way PPML gravity benchmark on the same Flows associates a Trade Agreement with only +5% (95% interval −0.4% to +11%), against the model's +57% Baseline-weighted mean effect and a literature range of +21% to +100%: the model's size is plausible against published estimates but about eleven times what pair-level variation over 2012–2024 shows. Section 7 shows that these associations are several times what a real agreement delivered, so they are an upper-end order of magnitude, never a forecast.

## 7. Evaluation: accuracy on the 2023 test year

The course's **time split** is the honest way to evaluate a model that will be used on later data: fit on the past, test on a year the model never saw. Here the years are train 2012–2021, validate 2022 (the hyperparameter search of section 5) and test 2023. The Baseline models of section 5 were refit on 2012–2023, so they cannot be tested on 2023. The evaluation therefore trains five **test-year fold models** on 2012–2022 with the configuration and round count chosen on 2022, and scores 2023 with them. Nothing about 2023 shaped tuning or training: the pipeline asserts that the test year is not among the tuning years, and a test (`test_the_test_year_is_untouched_during_tuning`) multiplies every 2023 Flow by 1,000 and checks that the search results do not move. This is the course's **leakage** rule applied to model selection, not only to preprocessing.

Following the course's **baselines and metrics** advice (report the naive baseline first), every metric is computed for the model and for a **naive baseline**: the Exporter's share of the Product's world exports times the Importer's imports of that Product. It is the "no fundamentals" prediction a trade analyst would make on a spreadsheet, and like the model's features it leaves the Flow itself out.

The course distinguishes the **loss** a model minimises from the **metric** people read. We report one of each kind and one that goes beyond the notes:

| Metric | Kind | Why |
|---|---|---|
| Tweedie deviance (power 1.5, weighted) | the training loss | measures the fit on every row, zeros included; sampled zeros count by their weight |
| RMSE of log(1 + v), non-zero Flows | the course's RMSE, on a log scale | the typical error in orders of magnitude; raw-dollar RMSE would be ruled by a handful of huge Flows |
| Spearman rank correlation within each Product | a ranking metric (beyond the notes) | Advisors read the model as a ranking of Markets for one Product, so we score exactly that, averaged over Products |

Two scopes are evaluated. **All Exporters** scores every 2023 training row (every non-zero Flow plus the sampled zero cells): UK rows out-of-fold, other rows by the mean of the five fold models. **UK out-of-fold** scores the UK's full 2023 Product × Market grid, each pair by the fold model that never saw it (ADR 0001), which is how the Baseline is produced. As in section 5, the models train on every UK row and a 58.9% reweighted sample of non-UK rows on real data. The fixture keeps every row.

The headline is the within-Product rank correlation for UK pairs, phrased for an Advisor:

In [ ]:
from exportspredictor import evaluation

report = bundle.evaluation
print(report["headline"]["text"])
{key: report[key] for key in ("test_year", "test_training_years", "validation_year", "non_uk_sampling_rate", "evaluation_seconds")}

In [ ]:
reporting.ranking_by_version(bundle)

### How the model was chosen: the experiment loop

The served Model Form was chosen by the experiment loop (`exportspredictor.loop`, ADR 0008) on one Selection Measure in the validation year, with an Arm kept only if it beat the best so far by more than the Win Threshold. The artefact records the loop's saved results; in real mode they are read from the loop's results file and checked against the artefact. The tables and the chart below are read from the artefact. An Arm marked `stopped_early` stopped within its first 50 rounds, so the stopping rule, not the configuration, decided its result.

In [ ]:
ranking_results = bundle.metadata["ranking_experiments"] if FIXTURE_MODE else loop.ranking_record(loop.load())
print("The artefact records the experiment loop's saved results:", ranking_results == bundle.metadata["ranking_experiments"])
with pl.Config(tbl_rows=20):
    display(reporting.experiments(bundle))

In [ ]:
reporting.experiments_chart(bundle)

The five-fold confirmation, fold by fold, of the best Arm against v2 (the last row is over every fold):

In [ ]:
reporting.confirmation(bundle)

Every metric, model against naive baseline, for both scopes:

In [ ]:
pl.DataFrame([{"scope": scope, **result["overall"]} for scope, result in report["accuracy"].items()]).drop("group", strict=False)

### Where the model is weak: Sectors and regions

The breakdowns are the course's *evaluating beyond the average: worst group*. They also feed the **bias review** of section 10. Here bias is about countries and Sectors, not people: BACI's data are weaker where countries report late or carelessly, and the model may be weaker for those Markets. Each table is sorted weakest first, by the model's within-Product rank correlation for UK pairs.

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_sector"]).head(10)

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_region"])

### Stability of the 2024 Baseline

BACI's 2024 data are provisional, so we check that the Baseline's rankings are not an artefact of one noisy year. The Baseline models also score the 2023 UK grid, and we compare the two years' rankings within each Product, and the overlap of the top 1,000 Opportunities. This is also the course's **drift** check in miniature: when BACI releases a new year, the same comparison shows how far the rankings moved.

In [ ]:
print(report["stability"]["text"])
report["stability"]

### The Potential Range: how much comparable pairs vary

An Export Gap is only worth an Advisor's time if it is larger than the ordinary scatter of trade around what the fundamentals suggest. Each UK pair therefore gets a **Potential Range**: the range within which Actual Exports fall 80% of the time for pairs with about the same Export Potential (ADR 0007). It describes how much comparable pairs vary, not how precisely Export Potential itself is known, so it is never shown as a confidence interval. An Opportunity whose Actual Exports lie inside its range is badged an **Uncertain Opportunity**; it stays in the ranking with a caveat.

This extends the course's **held-out validation** into **split conformal prediction**. A validation set normally gives one error figure for the whole model; here the held-out residuals are turned into a range for each prediction, with a coverage guarantee that rests only on the calibration rows being exchangeable with the rows we apply it to. Three choices make it honest:

- **Calibration rows are other Exporters' 2024 Flows and Zero Cells, never the UK's.** A UK residual is its Export Gap with the sign flipped, so a range calibrated on UK residuals would only measure "a typical UK gap" and call every gap normal. The fold models train on years before 2024, so these residuals are out of sample, and each row is scored by the fold model its hash (Exporter, Product, Market) assigns it to, as UK pairs are.
- **Mondrian bands.** Tweedie errors grow with the prediction, so one pair of offsets for all sizes would be far too wide for small pairs and too narrow for large ones. The rows are cut into 20 equal-count bands of Export Potential (each with at least 1,000 rows, or merged with a neighbour), and each band's 10th and 90th percentiles of signed residuals are its offsets. The range is clipped at $0.
- **Coverage is checked, not assumed.** The bands are refitted on one half of the calibration rows (split by a hash of the pair) and the other half is scored against them: overall, per band and per Exporter. Exchangeability between the UK and other Exporters is assumed, so the per-Exporter spread is the evidence that it roughly holds.

The 2024 offsets are applied around the Gap Period mean Export Potential and compared with the Gap Period mean Actual Exports. A three-year mean is less noisy than one year, so the range is wider than strictly needed and errs towards flagging more Opportunities as uncertain, never fewer.

In [ ]:
from exportspredictor import conformal

potential_range = report["potential_range"]
print(conformal.coverage_text(potential_range))
bundle.potential_range_bands

Coverage on the held-out half, per band of Export Potential (the bands refitted on the other half):

In [ ]:
pl.DataFrame(potential_range["by_band"])

And per Exporter, lowest first: an Exporter far below 80% is one the exchangeability assumption fits badly.

In [ ]:
pl.DataFrame(potential_range["by_exporter"]).sort("coverage")

The largest UK Opportunities with their Potential Ranges, and whether each is an Uncertain Opportunity:

In [ ]:
baseline.filter(pl.col("export_gap_usd") > 0).sort("export_gap_usd", descending=True).select(
    "hs6", "iso3", "export_potential_usd", "actual_exports_usd", "potential_range_low_usd", "potential_range_high_usd", "uncertain_opportunity"
).head(10)

On the fixture the calibration set is tiny (about 60 rows in three bands), so the held-out coverage lands only roughly on 80% and the per-band and per-Exporter figures swing widely; they show the check working end to end, nothing more.

On the real data (artefact version `20261118T042154Z`) the bands come from 12.8 M rows of 99 other Exporters' 2024 Flows and Zero Cells, scored in 172 s on the CPU. All 20 bands hold about 640,000 rows, so none merged.

- **Coverage is 85.8% on the held-out half, against the 80% target.** The range errs on the wide side, and almost all of the excess sits in the five smallest bands (Export Potential under about $1,200), where coverage is 84–90%. There most Actual Exports are exactly $0, so many residuals tie at one value and the 10th and 90th percentiles enclose more than 80% of them. From band 6 upwards, where Opportunities of any size live, coverage is between 79.6% and 80.3% in every band.
- **By Exporter the lowest coverage is China's, 78.6%**, then the USA, Türkiye, Germany and Italy at 80–81%. These large, diversified Exporters are the closest comparators to the UK, and they sit at the target. So the exchangeability assumption that carries the range over to UK pairs looks reasonable for the pairs that matter, though it stays an assumption: no UK residual is ever used to check it.
- **87% of Opportunities (917,556 of 1,055,772) are Uncertain Opportunities, but none of the top 1,000 by Export Gap is.** Most positive gaps are the small gaps of *New Market* pairs described in section 5, well inside the ordinary scatter, and the badge says exactly that. The largest gaps clear it. So the badge works as intended: it marks the long tail as noise and leaves the head of the ranking unbadged.
- **But the range has a fixed width in dollars within each band, and that limits what the badge means for large pairs.** The offsets are dollar amounts, not proportions, so every pair in the top band (Export Potential from about $2.29m upwards) gets the same range, from $10.3m below to $12.5m above its Export Potential, about $22.9m wide whatever its size. For turbo-jet parts to France (Export Potential about $1.4bn) that is a range of ±1%, a sliver on the Why? page; for a $3m pair it is several times the pair. So for large pairs "Uncertain" just means an Export Gap under about $10.3m, and "not Uncertain" says nothing about how large the scatter of a $1bn pair really is. This is part of why none of the top 1,000 is badged. Tweedie errors grow with the prediction, so a range relative to Export Potential (offsets on a log scale, or more bands at the top) would be the better design for a later version. The model card states this limitation.

### Known limitations

These are shown on the app's Model Evaluation page, so that Advisors can caveat their briefings. The last one came out of section 5's sanity check of the top Opportunities.

In [ ]:
pl.DataFrame(evaluation.LIMITATIONS)

### Results on the real data

These are the results of the v2 run of 18 Nov 2026 (artefact version `20261118T042154Z`), the first time 2023 was read for evaluation. The five test-year fold models took 73 minutes to train on the RTX 3080. Scoring the 14.3 M test rows and the 1.16 M-pair UK grid took 11 minutes on the CPU. Excluded Products are left out of every metric.

| Scope | Metric | Model | Naive baseline |
|---|---|---|---|
| All Exporters (14.3 M rows) | Tweedie deviance | **356** | 4,334 |
| | RMSE of log(1 + v), non-zero Flows | **2.72** | 3.34 |
| | Spearman within Product | **0.76** | 0.57 |
| UK out-of-fold (1.16 M pairs) | Tweedie deviance | **569** | 2,263 |
| | RMSE of log(1 + v), non-zero Flows | **2.38** | 2.73 |
| | Spearman within Product (headline) | 0.51 | **0.52** |

How to read these numbers:

- **Across all Exporters the model clearly beats the naive baseline** on every metric. Its deviance is about a twelfth of the baseline's, and its within-Product ranking improves from 0.57 to 0.76.
- **For UK pairs the model does not beat the naive baseline on the headline.** Its within-Product rank correlation is 0.51 against the baseline's 0.52, and the app's headline says so in those words rather than claiming a win. Two reasons are plausible:
  - Out-of-fold scoring removes exactly the pair-specific information (the UK's own past exports to that Market) that would make UK rankings easy. That is the point of ADR 0001.
  - The UK's world market share in a Product, times each Market's imports, is already a strong ranking of its own Markets. The model's features hold much the same information, so on ranking alone it has little to add.
- **The model is much better at the *size* of UK Flows.** Its deviance is a quarter of the baseline's, and its RMSE of log value is lower (2.38 against 2.73). So Export Potential is a better estimate of how large a Flow should be, even where the model ranks Markets no better than market share does. That is the quantity an Export Gap is built from.
- **No test row has an observed tariff.** Tariff data end in 2021, so every 2023 row, in both scopes, uses an Applied Tariff carried forward from 2021 (`tariff_imputed`). The split by tariff source the evaluation reports is therefore all "imputed" and the "observed" group is empty: the cost of carrying tariffs forward cannot be measured on this test year, only assumed small.
- **Weakest Sectors for UK pairs** are ores (chapter 26, rank correlation 0.20), fish (03, 0.21), furskins (43, 0.22), live animals (01, 0.22), wood pulp (47, 0.25) and silk (50, 0.26). In each the naive baseline does better. Most are commodities where sanitary rules, quotas and logistics decide Markets, and the model sees none of these.
- **Weakest regions for UK pairs** are Latin America & Caribbean (0.32) and Sub-Saharan Africa (0.34). Small territories BACI leaves unclassified score worst (0.20). North America is the strongest (0.82). The model beats the baseline in Europe & Central Asia (0.66 against 0.56) and South Asia (0.65 against 0.55), but not in East Asia & Pacific (0.59 against 0.61) or Sub-Saharan Africa. This matches the bias concern in section 10: the model is weakest where Markets are small, distant or poorly reported, so Opportunities there need more checking.
- **Stability.** Scored by the same Baseline models, each Product's 2024 ranking of Markets has a rank correlation of 0.78 with its 2023 ranking, and 60% of the top 1,000 Opportunities are the same in both years. Rankings are broadly stable, but the provisional 2024 data move the top of the list noticeably. This is one more reason the Export Gap averages the Gap Period instead of using 2024 alone.

### Lift: do large Export Gaps point at growth?

Accuracy says how well the model reproduces Flows in a year it never saw. An Advisor needs something else: when the model says the UK underperforms in a pair, does that pair tend to grow later? We cannot test that on UK pairs with the Baseline models, because the future after 2024 has not happened. So we replay the past (`backtests.lift_backtest`):

1. **Hold out the Exporters v1 held out**, read from the frozen v1 tables and kept only where they are in v2's Exporter scope, so v1 and v2 are scored on the same countries (v1 chose 20% of Exporters by a hash of their codes). They play the UK's role: just as each UK pair is scored by a fold model that never saw it (ADR 0001), these countries' Flows are never trained on.
2. **Train one model on 2012–2017** with the tuned configuration, round count and 58.9% non-UK sample of section 5.
3. **Score the held-out Exporters' pairs in 2015, 2016 and 2017** and rank them by their Export Gap over that Gap Period: the mean of the three yearly gaps, as the Baseline's Export Gap is the mean over 2022–2024. The pairs are the sampled 2017 rows; a sampled Zero Cell with no row in an earlier year had no Flow then, and its Export Potential is the mean over the years it has.
4. **Look up 2023.** A pair *grew materially* if its 2023 Flow was at least 50% and at least $100,000 above its 2015–2017 mean Actual Exports (so a pair that appears from nothing counts once it reaches $100,000). Sampled zero cells count by their weight.

**Lift** is the growth rate of the top tenth of pairs by Export Gap divided by the growth rate of all pairs. The course meets lift as a gate number for Grad-CAM ("share of attention on ink ÷ share of image that is ink", with a gate at lift ≥ 1); here it plays the same role for Opportunities: above 1 means the ranking concentrates growth. Two things go beyond the notes:

- **A confidence interval from a cluster bootstrap.** The course meets the bootstrap inside random forests. Here it resamples whole Exporters 1,000 times, because one country's pairs rise and fall together (a currency, a recession), so resampling pairs would overstate the precision.
- **A size-matched Lift.** Large gaps belong to large pairs, and a $100,000 rise is easier for a large pair. So we also compare the top tenth with pairs in the same Export Potential tenth. If that ratio stays above 1, the gap says something beyond size.

Lift is a tendency across many pairs: it does not promise that any one Opportunity will grow.

In [ ]:
lift = report.get("lift")
print(f"Lift: {NOT_RUN}." if lift is None else lift["text"])
None if lift is None else pl.DataFrame(lift["by_decile"]).rename({"decile": "Export Gap tenth (1 = largest)", "rate": "share that grew", "pairs": "weighted pairs"})

In [ ]:
reporting.lift_by_tenth(bundle)

### Event Backtests: Scenario Effects against a real Trade Agreement

A Scenario Effect is an association learned from many agreements (ADR 0002), so it should be checked against one that really happened. The **EU–Japan Economic Partnership Agreement** came into force on 1 February 2019. `backtests.event_backtest` does the following:

1. **Train one model on 2012–2018 only.** It never reads a year from the event onwards: `train_backtest_model` refuses later years, and a test (`test_no_event_backtest_model_sees_data_from_after_its_event`) multiplies every Flow from the event year on by 1,000 and checks that the trained model is byte-for-byte the same.
2. **Predict 2019–2021** EU→Japan and Japan→EU Flows with the agreement on and off. This is the course's *counterfactual test* (flip one attribute, hold the rest), used here as a measurement, not a fairness check. The difference is the model's Scenario Effect.
3. **Measure what happened** with a **difference-in-differences**, a standard method from econometrics that is not in the course notes: the change in EU exports to Japan from 2016–2018 to each later year, divided by the change in EU exports to the **Control Markets** over the same years. The Control Markets are the ten largest high-income Markets that had no Trade Agreement with any EU member at any point in the window. A 95% interval comes from resampling Products.

**Caveat.** COVID-19 disrupted trade in 2020 and 2021, and not evenly across Markets, so 2019 is the cleanest year. The method assumes EU–Japan trade would otherwise have moved like EU trade with the control Markets, and values are nominal USD, so exchange-rate swings (the yen against the dollar) move them.

A secondary check uses the **UK–Australia and UK–New Zealand** agreements (in force 31 May 2023). The test-year fold models of this section were trained on 2012–2022, before them, so they score it, each UK pair by the model that held it out. Only 2024 serves as a post year, and 2024 is provisional.

In [ ]:
backtest = report.get("event_backtest")
print(f"Event Backtests: {NOT_RUN}." if backtest is None else backtest["text"])
reporting.event_backtests(bundle)

In [ ]:
None if backtest is None else {
    (event["name"], direction["label"]): len(direction["controls"])
    for event in backtest["events"]
    for direction in event["directions"]
}  # the number of control Exporters

### Backtest results on the real data

From the v2 run of 18 Nov 2026 (artefact version `20261118T042154Z`). The Lift model (2012–2017) and the EU–Japan model (2012–2018) were trained with the tuned configuration, its 1,453 rounds and the 59% non-UK sample of section 5. Together with the Event Backtests they took 23 minutes on the RTX 3080. They are cached under `data/derived/backtest_models/`, so a re-run (`python -m exportspredictor.backtests`) only scores.

**Lift.** v1 held out 45 Exporters; 13 of them are inside v2's top-100 scope (the UAE, Australia, Bahrain, Colombia, Denmark, Iraq, Nigeria, Oman, Qatar, Serbia, Slovakia, Uganda and Venezuela), so the Lift is measured on those 13. They have 1.19 M sampled 2017 pairs (8.6 M once sampled zeros are weighted). Of the top tenth by Export Gap over 2015–2017, 6.1% grew materially by 2023, against 1.2% of all pairs: a **Lift of 5.2 (95% interval 3.6 to 9.5)**. The size-matched comparison changes the reading. Pairs in the same Export Potential tenth grew 7.7% of the time, so the **size-matched Lift is 0.79 (0.72 to 0.86)**. The largest gaps point at the pairs most likely to grow, but mostly because they are large pairs. Among pairs of similar size, a large gap was, if anything, a slightly weaker sign of growth. The growth share falls steeply by gap tenth: 6.1%, then 0.8% and 0.2%, and almost nothing in the middle tenths. For Advisors this means an Opportunity marks a Market worth a look, not one more likely to grow than its peers of the same size. The Model Evaluation page says so in those words. v1 showed the same pattern (Lift 6.6, size-matched 0.83, on all 45 Exporters), but the two are not directly comparable: the v1/v2 comparison scores both on the same 13 Exporters.

**EU–Japan Event Backtest.** The model associates the agreement, with the tariff zeroed in the "on" state and the MFN rate restored in the "off" state, with **+35%** in EU Export Potential to Japan over 2019–2021 (+35%, +35%, +36% by year). The observed difference-in-differences against the Control Markets (USA, Russia, UAE, Australia, Hong Kong, Saudi Arabia, Taiwan, Qatar, Kuwait, New Zealand) is **+2% (95% interval −7% to +14%)**. It is +5% in 2019, the year COVID cannot explain. For Japan's exports to the EU the model says **+42%** and the observed change is **−0% (−10% to +10%)**. So the model's Scenario Effects are again **several times larger than the change actually observed**, well outside the interval. The reasons are plausible but untested here:
- The model learned from all agreements in 2012–2018, many of them between countries whose trade was growing anyway.
- The agreement flag also absorbs closeness between partners that no other feature records.
- The EU–Japan agreement phased in tariff cuts over years, so its full effect lies beyond 2021. Zeroing the tariff at once in the "on" state overstates the early years by construction.

**UK–Australia/New Zealand (secondary).** The test-year fold models say **+31%** for 2024. Observed is **−16% (−27% to −1%)** against Control Markets (excluding Russia and Belarus, under sanctions). With one provisional post year, this only confirms the direction of the EU–Japan result.

**What this means.** The monotone constraint guarantees that an agreement never lowers Export Potential (ADR 0002), and the tariff coupling makes an agreement move the tariff with it (ADR 0006). Neither makes the size right. Scenario Effects should be read as an upper-end association, about the order of magnitude, not a forecast. The Scenarios page states the EU–Japan result next to every Scenario, so Advisors see this before they quote a number.

### v1 against v2: the pre-registered comparison

v2 changes the scope, the sampling, the features and the tuning, so a fair question is whether it is better than v1 at all. The comparison (`comparison.compare`) was written and tested on a synthetic frozen v1 *before* 2023 was read for it, and its rule was fixed in advance (decision #38). That is the course's warning about choosing a test after seeing the data, applied to the model comparison itself: the conclusion cannot be picked to suit the results.

- **v1 is never re-run.** Its predictions come only from the frozen tables written before any v2 code landed (`freeze`). Their manifest records the `v1` tag, the commit and a SHA-256 checksum per table, and the comparison refuses a table that no longer matches it.
- **Same rows for everyone.** v1, v2 and the naive baseline are scored on exactly the same keys: every UK 2023 pair outside the Excluded Products, each scored out-of-fold by both models (ADR 0001), and v1's own 2023 test rows (every positive Flow plus v1's weighted Zero Cell sample) restricted to v2's Exporter scope and non-Excluded Products.
- **Primary metric:** the UK out-of-fold within-Product rank correlation, with a **paired bootstrap** over Products of v2 − v1 (course: resampling for uncertainty; pairing removes the variation the two models share, so the interval is narrower than comparing two separate intervals).
- **Secondary metrics:** Tweedie deviance and RMSE-log on the all-Exporters rows (paired bootstrap over Products), the size-matched Lift on the same held-out Exporters, and the Event Backtest distance: how far each model's Scenario Effect lies from the observed difference-in-differences in the event's first year. Later years are shown with the COVID caveat but do not count.

**The rule.** "v2 beats v1" only if (1) the bootstrap interval of the primary metric lies above 0, (2) v2 beats the naive baseline by more than v1 did, and (3) no secondary metric gets worse beyond its own interval. Otherwise the result is mixed and is reported metric by metric, in associational language. The comparison lives in the evaluation results and the model card, and the API does not serve it.

In [ ]:
from exportspredictor import comparison

v1_comparison = report.get("v1_comparison")
if v1_comparison is None:
    print("This artefact has no v1 comparison: it is run once, with the final evaluation on 2023.")
else:
    print(v1_comparison["verdict"]["text"])
    print({name: rows for name, rows in v1_comparison["rows"].items()})

Every metric of the comparison, the primary first. `counts` marks the metrics the rule reads; `worse` marks a secondary metric that got worse beyond its own interval:

In [ ]:
None if v1_comparison is None else pl.DataFrame(
    [
        {
            "metric": v1_comparison["primary"]["label"],
            "v1": v1_comparison["primary"]["v1"],
            "v2": v1_comparison["primary"]["v2"],
            "difference": v1_comparison["primary"]["difference"],
            "counts": True,
            "worse": False,
        }
    ]
    + [
        {
            "metric": entry["label"],
            "v1": entry["v1"],
            "v2": entry["v2"],
            "difference": entry["difference"],
            "counts": entry["counts"],
            "worse": entry["worse"],
        }
        for entry in v1_comparison["secondary"]
    ]
)

v1 is also measured on every Product it scored, Excluded Products included, against the common rows. The difference is what the exclusions alone change, before any modelling change:

In [ ]:
None if v1_comparison is None else pl.DataFrame(
    [
        {"rows": name, "measured on": measured_on, **v1_comparison["v1_all_products"][name][measured_on]}
        for name in comparison.ROW_SETS
        for measured_on in ("common_rows", "all_products")
    ]
)

### The gain ladder: where the change came from

v2 changed several things at once, so the verdict alone cannot say which change moved the primary metric. The **gain ladder** adds two more 2023 test fold models and climbs from v1 to v2 one group of changes at a time (spec #39, decision #38):

- **v1**, as frozen;
- **A**: v1's 22 features and configuration (depth 8, variance power 1.3, 1,491 rounds), trained on v2's rows: the top-100 Exporter scope, v2's sampling rates and no Excluded Products;
- **B**: A plus v2's new features (Relatedness, the re-export ratio, the Applied Tariff);
- **C**: v2 itself, with its tuned configuration.

Every rung is scored on the same UK common rows, out-of-fold, and measured on the same Products as the primary metric. This is the course's ablation idea run in the other direction: instead of removing one input from a finished model, each step adds one group of changes to the last. The steps are cumulative, so **the order of the steps affects how the change is split**: adding the features before the new scope could credit them with a different share (a Shapley-style split over every order would remove that, at the cost of more models). The ladder is for reporting only and decides nothing: the verdict above is the pre-registered rule's alone.


In [ ]:
ladder = None if v1_comparison is None else v1_comparison.get("ladder")
None if ladder is None else pl.DataFrame(ladder["rungs"]).join(
    pl.DataFrame(ladder["steps"]).select(
        name="to", step="label", change="change", ci_low=pl.col("ci").list.first(), ci_high=pl.col("ci").list.last()
    ),
    on="name",
    how="left",
)


The waterfall below starts from v1's primary metric, adds each step's change, and ends at v2's. Blue steps raise the metric and red steps lower it; the grey bars are v1 and v2 themselves, and the dashed line is the naive baseline on the same rows. The axis is cut to the range of the values, so the bars' heights show the changes, not the metric from 0.


In [ ]:
reporting.gain_ladder(bundle)

**The real comparison on 2023** (artefact `20261118T082751Z`, run once, nothing changed afterwards).

**Verdict under the pre-registered rule: v2 does not beat v1; the result is mixed.** Two of the three conditions fail:

- **Primary metric.** On the same 1,156,050 UK pairs (5,073 Products ranked by both models), v2 ranks Markets with a within-Product rank correlation of **0.511 against v1's 0.540**, a difference of **−0.028 (95% paired-bootstrap interval −0.029 to −0.027)**. The interval lies wholly below 0. It is narrow because it resamples over 5,000 Products; it measures how much the result depends on which Products were drawn, not how much it would move with another training seed.
- **Margin over the naive baseline.** The naive ranking scores 0.523 on these rows: v1 was 0.017 above it, v2 is 0.012 below it.
- **Secondary metrics: none worse beyond its interval.** On all Exporters' rows v2 estimates the *size* of a Flow better: Tweedie deviance 355 against 401 (interval of the difference −50 to −43) and RMSE-log 2.72 against 2.84. Size-matched Lift is 0.79 against v1's 0.83 (inside v1's interval, 0.79 to 0.88); raw Lift, not counted, 5.2 against 5.8. In the EU–Japan Event Backtest's first year (2019), v2 lies closer to the observed difference-in-differences for EU exports to Japan (distance 0.30 against 0.39) and further for Japan's exports to the EU (0.38 against 0.30), within the tolerance of the observed interval. Neither model lands inside the observed interval in any year: both associate the agreement with changes of +34% to +46% across 2019–2021, against observed changes between −3% and +5%.

**v1 on all Products.** v1 ranks UK Markets at 0.538 on every Product it scored and at 0.540 on the common rows, so leaving out the Excluded Products changes the measurement by only 0.002. The exclusions in the *evaluation* rows are not why v2 ranks worse.

**The gain ladder** (reporting only) associates almost all of the drop with the first step: v1's own features and configuration, trained on v2's rows (top-100 Exporter scope, every UK Zero Cell kept, 5% of other Zero Cells, about 59% of other rows, no Excluded Products), rank at 0.519, **−0.020** below v1. v2's new features then go with **−0.004** and the tuned configuration with **−0.004** more. With this order, the change in *training rows* carries most of the loss; another order could split it differently. The ladder cannot say which part of the row change matters: the scope, the UK Zero Cells kept at full weight, or the sampling rates were changed together. One reading, untested here, is that every choice in v2 was made on the 2022 Tweedie deviance, which v2 improves, while the UK ranking metric was only a guardrail (course: a model is optimised for the loss it is given, and the evaluation metric can disagree with it). Nothing was retuned after reading 2023: doing so would turn the test year into a validation year.


**Conclusion.** The evaluation was set up so that it could fail, and in places it did. On 2023, read here for the first time, the model clearly beats the naive baseline across all Exporters. For UK pairs scored out-of-fold it does not beat the baseline's ranking (0.51 against 0.52); its value there is the much better estimate of the *size* of a Flow (a quarter of the baseline's deviance). Every 2023 tariff is carried forward from 2021, so what that costs could not be measured. Large Export Gaps concentrate growth (Lift 5.2), but no more than pairs of the same size do (0.79). The Potential Range covers 86% against an 80% target, close to 80% for every band above a few thousand dollars, and it badges the long tail of small gaps as uncertain while leaving the top of the ranking clear. Scenario Effects overstate the change seen after real agreements several-fold. The app states each of these findings next to the numbers they qualify (course: *no signal is a result, not a failure*), and the model card in section 10 records them. Against v1, under the rule fixed before 2023 was read, v2 does **not** win: it estimates the size of Flows better on every Exporter's rows, but it ranks UK Markets less well than v1 did (0.511 against 0.540) and falls below the naive baseline that v1 beat. The gain ladder associates most of that loss with the new training rows rather than the new features or the tuning.


## 8. Explaining Export Potential

An Advisor who takes an Opportunity to a firm must say *why* the model expects more trade. The course distinguishes **global** explanations (what drives the model overall) from **local** ones (why this prediction), and recommends **SHAP** for local explanations that are additive and faithful to the model being shipped. Both are computed here.

XGBoost computes exact TreeSHAP values itself (`pred_contribs=True`), so no approximation or background sample is needed. Each pair is explained by **its own fold model**, the one that held it out and scored its Baseline (ADR 0001): explaining any other model would explain a number the Advisor never sees (course: *explain the model you ship*). The model has a log link, so contributions add up on the log scale, and `exp` of the base value plus every contribution reproduces the Baseline Export Potential exactly (`tests/test_explain.py` checks this for every fixture pair; on the real artefact the error is below 3e-5 over 200 pairs). On the dollar scale each factor multiplies Export Potential by `exp(contribution)`.

**Global importance** is the mean absolute contribution over a random sample of 20,000 Baseline pairs. Unlike impurity importance, it is measured on the shipped predictions and cannot rank a feature the model never uses (course: *feature importance: impurity vs permutation*).

In [ ]:
from exportspredictor import explain

importance = explain.global_importance(bundle, sample=20_000)
importance.with_columns(typical_multiplier=pl.col("mean_abs_contribution").exp())

On the real artefact (20,000 pairs, a few minutes on the CPU with the depth-12 fold models), demand dominates: how much the Market imports the Product from others moves a typical prediction by a factor of about 8. Then come UK supply of the Product to other Markets (×2.8), the high-income share of the Market's imports (×1.8), distance (×1.6), the Market's GDP (×1.6) and UK specialisation (×1.4). Two v2 features follow closely: the Market's re-export ratio (×1.27) and the Applied Tariff (×1.24). The tariff matters more on average than the Trade Agreement flag (×1.15), and Relatedness adds about 8%. The bloc flags matter much less: CPTPP and EFTA move a typical pair by about 1%. This is expected, since most pairs' policy status never changes, but it also shows that Scenario Effects rest on a small part of the model.

Low global importance is also a governance signal (course: mean |SHAP| *supports minimisation*). The EFTA and CPTPP flags and the common-colonizer feature contribute about 1% to a typical prediction. They stay in the model because Scenarios need the bloc flags, but a future version could drop common colonizer at no cost.

**A local explanation.** The largest Opportunity on the real data is parts of turbo-jets and turbo-propellers (`841191`) to France. The Why? page shows the same factors in the same words:

In [ ]:
top_opportunity = baseline.sort("export_gap_usd", descending=True).row(0, named=True)
why = explain.explain_pair(bundle, top_opportunity["hs6"], top_opportunity["iso3"])
print(f"{top_opportunity['description'][:60]} to {why.market}: Export Potential {explain.usd(why.export_potential_usd)}, scored by fold model {why.fold}")
display(pl.DataFrame([{"factor": f.label, "value": f.value_text, "multiplier": f.multiplier} for f in why.factors[:8]]))
pl.DataFrame([{"year": point.year, "actual_exports_usd": point.actual_exports_usd} for point in why.history])

On the real data the model's 2024 Export Potential for UK turbo-jet parts in France is **US$2.5bn**, scored by fold model 2, against Actual Exports of $0.08bn in 2024. The explanation reads as an Advisor would want. The UK exports $12.5bn of this Product to other Markets (×28) and France imports $2.5bn of it from others (×21). The UK is highly specialised in it (8.6 times the world share, ×2.2), and France is close (344 km, ×2.1). It also shows a reason for caution: France *re-exports* 378% of what it imports of these parts (×1.36). France is home to large aero-engine and aircraft makers, so much of this trade runs inside supply chains that existing contracts decide.

The history adds what the model cannot see. UK exports to France have stayed near $0.1bn every year since 2019, while the yearly Export Potential rose from $0.58bn (2022) to $1.09bn (2023) and $2.47bn (2024). The Gap Period mean Export Gap ($1.28bn) is the figure the ranking uses, and the 2024 jump rests on provisional data. An Advisor should check the supply-chain reasons before pitching this as an Opportunity.

Two limits apply to every explanation. SHAP values are **associations inside the model**, not causes: "distance lowers Export Potential" means comparable distant pairs trade less. And several top factors (the Market's size, its imports, distance) are not things an Advisor can change, which the course flags as a test of *actionable* explanations. The actionable reading is the supply side: the UK already sells this Product well elsewhere.

**Conclusion.** Every Export Potential can be decomposed exactly into named factors from the fold model that produced it, in words an Advisor can repeat. Globally the model is driven by what Markets buy and what the UK sells; policy features carry a small part of each prediction, which is consistent with Scenario Effects being sensitive to how the model learned them (section 7). Local explanations combined with the Actual Exports history are the Advisor's main defence against a misleading Opportunity.

## 9. Saving the artefacts

The API must serve exactly what was evaluated (course: *registry versions* and *reproducible artefacts*). In real mode the artefact was trained and saved by `python -m exportspredictor.pipeline`, and this notebook reads it; the fixture run saves its own. Either way it is a versioned artefact folder with the five fold models, the Baseline table, the Baseline's model inputs and folds, the Actual Exports history, the Product and Market lookups, the evaluation report of section 7 (`evaluation.json`), the model card of section 10 (`model_card.md`), and a metadata file (sources, sampling rates, the search, the chosen parameters, device and timings).

In [ ]:
# The real artefact was saved by the pipeline step; the fixture run saves its own.
saved_to = artefacts.save(bundle, artefacts_dir()) if FIXTURE_MODE else artefacts_dir() / bundle.version
saved_to

**Conclusion.** Each run writes a new folder named by its UTC timestamp, and the API loads the newest complete one (the metadata file is written last, so a half-written folder is never loaded). Rolling back means removing or renaming the newest folder. Steps that add results to a trained model (Blocked Opportunity values, the evaluation, the backtests) copy the folder to a new version rather than editing it in place, and record `copied_from`, so every version traces back to the training run that produced its models.

## 10. Governance

The course treats responsible AI as **design constraints with owners and evidence**: "a commitment without an artefact is an aspiration". This section produces those artefacts for the model: a model card generated from the evaluated artefact, data lineage, a bias review across regions and reporting quality, the regulatory classification under the EU AI Act and the UK's approach, and the drift checks and yearly refresh that keep the model honest after release.

### Model card

The course asks for a short structured card covering intended and out-of-scope use, owner, metrics by group, limitations and data sources, *generated from code and versioned with the model*, printing UNASSIGNED rather than omitting a section. `governance.model_card` builds it from the artefact's metadata and evaluation report, so its numbers are the evaluated numbers. It also cites, in substance, the decisions a reader of the model needs (ADRs 0001–0007 and the decisions behind v2: the ablation, the TPE search, the Gap Period, the pre-registered comparison and the PPML benchmark). Nothing about the training data is typed into the card by hand: the Exporter scope and both sampling rates are read from the artefact's metadata (ADR 0004), the dropped feature from the ablation results it carries, and the Excluded Products from its `excluded_products` table. Its limitations add, to the evaluated ones, the Potential Range's fixed width in dollars and the gap between the model's Trade Agreement Scenario Effect and the PPML benchmark of section 6, read from the benchmark saved with the artefact (`ppml.add_benchmark`). `write_model_card` saves it as `model_card.md` in the artefact folder, next to the models it describes.

In [ ]:
from IPython.display import Markdown

from exportspredictor import governance

# The fixture run writes its own card; the real card is written with its artefact and only read here.
card_path = (
    governance.write_model_card(bundle, saved_to) if FIXTURE_MODE else saved_to / governance.MODEL_CARD_FILE
)
Markdown(card_path.read_text(encoding="utf-8"))

The owner is **UNASSIGNED** on purpose: this is a course project, and naming who answers for the model is a decision for the team that adopts it. The course's warning applies: governance usually fails "at the name, not the technology".

### Data lineage

Every number the API serves should trace back to exact inputs (course: *data lineage and reproducible artefacts*). The lineage has three links:

1. **Raw downloads.** `governance.data_lineage()` fingerprints every file under `raw/` with its size and SHA-256, so a changed download is detectable. That includes the 1,294 UNCTAD TRAINS files behind the Applied Tariff (one per reporter and year, 2012–2021, with the WITS manifest) and the UN Statistics Division HS 2017 and HS 2007 → HS 2012 concordances that put them on BACI's revision. The source URLs and versions are in each package module (`baci.BACI_URL`, `gravity`, `policy`, `tariffs`, `hmrc`), and the TRAINS pull date is in the artefact's `sources`.
2. **Derived data.** Every derived table is rebuilt from the raw files by one `python -m exportspredictor.<module>` command, in the order of `scripts/rebuild.py`; none is edited by hand.
3. **Artefacts.** The metadata records the BACI release, the Exporter scope, the zero-sampling rates (other Exporters' and the UK's) and seed, the non-UK sampling rate, the ablation and learning curve, the search, the chosen parameters, the XGBoost version, the device and the timings, and `copied_from` links each version to the one it came from.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world is generated in code (exportspredictor.fixture), so there are no raw files to fingerprint.")
else:
    lineage = governance.data_lineage()  # hashes about 2.8 GB of downloads, under a minute
    display(lineage.group_by("source").agg(files=pl.len(), bytes=pl.col("bytes").sum()).sort("source"))
    display(lineage.filter(pl.col("source").is_in(["UNCTAD TRAINS", "UN HS concordances"])))
{
    **{key: bundle.metadata.get(key) for key in (
        "version", "source", "xgboost_version", "device", "zero_sampling_rate", "uk_zero_sampling_rate",
        "zero_sampling_seed", "non_uk_sampling_rate", "created_at", "copied_from",
    )},
    "exporters_in_scope": len(bundle.metadata.get("exporter_scope") or []),
}

On the real data the lineage lists 1,314 raw files, 2.8 GB in all: the BACI archive (1.27 GB), CEPII Gravity (1.46 GB), the 1,294 TRAINS files (61 MB), the two UNSD concordances, and the WTO, World Bank, IMF and HMRC downloads. The artefact chain runs from the v2 training run `20261118T042154Z` (fold models, Gap Period, Potential Range, evaluation and backtests in one pipeline run) to `20261118T082751Z`, which adds the v1 comparison and records `copied_from`; the API serves the latter. A gap remains: the artefact does not yet store the raw-file hashes themselves, so proving which download a version was trained on relies on the data folder not having changed since. Writing the lineage table into the metadata at training time is a small next step.

### Bias review: regions and reporting quality

Bias here concerns countries and Sectors, not people: the model holds no personal data. The course's two families still apply:

- **Data bias.** BACI is built from what countries report, and poorer countries report later, less completely and with larger errors. Absent Flows are read as zeros, so a poorly reporting Market looks as if it buys little (measurement and representation bias).
- **Algorithmic bias.** One model, one loss and one ranking serve all Markets, and the loss is dominated by large Flows, which are concentrated in high-income Markets (objective bias).

The review follows the course's **representation audit** (group sizes and denominators first, thin groups flagged) and **worst-group evaluation**, on the UK's 2023 out-of-fold grid. World Bank **income level** stands in for reporting quality, because no per-country reporting-quality score is published with BACI. A group is flagged *thin* when it has fewer than 500 UK Flows, the course's size floor.

In [ ]:
uk_accuracy = report["accuracy"]["uk_out_of_fold"]
bias_columns = ["group", "rows", "non_zero_flows", "actual_usd", "model_spearman_within_product", "naive_spearman_within_product"]
for breakdown in ("by_region", "by_income"):
    if uk_accuracy.get(breakdown):
        display(pl.DataFrame(uk_accuracy[breakdown]).select(bias_columns).with_columns(thin=pl.col("non_zero_flows") < 500))

On the real data (UK pairs, 2023, within-Product rank correlation, model against naive):

| Income level of the Market | UK pairs | UK Flows | Share of UK exports | Model | Naive |
|---|---|---|---|---|---|
| High income | 400,764 | 179,537 | 83% | **0.63** | 0.61 |
| Upper middle income | 298,004 | 81,534 | 13% | 0.43 | 0.43 |
| Lower middle income | 241,486 | 48,854 | 4% | **0.40** | 0.36 |
| Low income | 128,450 | 12,804 | 0.3% | 0.23 | 0.24 |
| Unclassified (small territories) | 87,346 | 2,947 | 0.04% | 0.20 | 0.02 |

What the review finds:

- **A steep gradient by income.** The model ranks Markets much better in high-income Markets (0.63) than in low-income ones (0.23). By region the same pattern shows: North America 0.82 and Europe & Central Asia 0.66, against Latin America 0.32 and Sub-Saharan Africa 0.34.
- **It is mostly a data effect, not the model treating poorer Markets worse.** Scored across *all* Exporters, the model's accuracy barely varies with the Importer's income group (0.69–0.75), and its gain over the naive baseline is largest for low-income Importers (0.69 against 0.15). The UK gradient comes from how little the UK sells there: in low-income Markets only 1 in 10 UK pairs has any trade, so most rankings are among near-zero Flows that BACI measures least well.
- **No group is thin by the course's floor**, but the low-income and unclassified groups together hold 0.3% of UK exports, so their metrics move with few Flows.
- **Who bears the errors.** An Advisor who reads the rankings at face value will see fewer, noisier Opportunities in low-income Markets. That could steer export support away from developing Markets for reasons of data quality rather than fundamentals.
- **v2's scope does not remove Markets.** The top-100 cut is on Exporters only (ADR 0004): every Market, low-income ones included, is still scored by models that trained on Flows into it.

**Mitigations in place:** the app's Model Evaluation page shows the region breakdown and the reporting-quality limitation, and every ranking can be filtered by region, so Advisors can compare Opportunities within a region rather than across them. A per-region or per-income threshold for Opportunities (course: *per-group thresholds*) would be the next step if the rankings were used to allocate support budgets.

**Potential Range coverage by Exporter.** The Potential Range carries a second fairness question. It is calibrated on other Exporters' rows and applied to the UK (ADR 0007), so it is only as fair as the assumption that their pairs vary like the UK's (exchangeability). That assumption cannot be tested on the UK without using the Export Gaps the model exists to find, but it can be tested *between* the calibration Exporters: if the range held for some and not others, it would also be unsafe to carry over. `governance.range_coverage_by_exporter` rates each Exporter's held-out coverage against the 80% target (within 5 percentage points is *near target*), the course's worst-group evaluation applied to an uncertainty estimate rather than to accuracy.

In [ ]:
range_coverage = governance.range_coverage_by_exporter(report["potential_range"])
display(range_coverage.group_by("status").agg(exporters=pl.len(), median_rows=pl.col("rows").median()))
range_coverage

On the real data, 34 of the 99 calibration Exporters are near the target and none falls below it by more than 5 points: the lowest is China at 78.6%, then the USA, Türkiye, Germany, Italy, Belgium, France and India, all at 80–81%. The other 65 are *too wide* (up to 94% for Iraq), and they are the small Exporters: their median is about 28,000 calibration rows against 106,000 for the near-target group, and across Exporters coverage falls almost perfectly with size (Spearman −0.98 against the log of the row count). Small Exporters' rows sit mostly in the smallest bands, where ties at $0 inflate coverage (section 7), so their excess is the same small-band effect, not a different kind of pair. For the UK this is reassuring rather than troubling: the large, diversified Exporters most like the UK are the ones at the target, and no group is under-covered, which would be the harmful direction (an Opportunity called certain when it is not). It stays an assumption. The model card lists the Exporters far from the target.

### Regulation: the EU AI Act and the UK approach

**EU AI Act.** The course's rule is that *intended purpose decides the tier*, not the technique. The intended purpose here is decision support for public export promotion: ranking country–product pairs and exploring policy associations for officials, from aggregate trade statistics.

- It is **not prohibited**: no manipulation, social scoring or biometric use.
- It is **not high-risk**: it does not fall in any Annex III use (employment, credit, education, essential services or benefits for natural persons, law enforcement, migration, justice, democratic processes, biometrics), and it is not a safety component of a regulated product. Its outputs concern countries and Products, never natural persons.
- **Transparency duties** (Article 50, from August 2026) target systems that interact with people as if human, or generate synthetic content. The app does neither, though it labels every number as a model output and states its limits, which meets the spirit anyway.
- So it is **minimal risk**, with no new legal duties. The 2026 Digital Omnibus moved high-risk dates (stand-alone uses to December 2027) but does not change this classification.
- The Act reaches providers placing systems on the EU market or whose outputs are used in the EU. A UK government tool used in the UK falls outside it, but classifying it is still the right discipline for UK government work with EU partners.

**Function creep would change the tier.** Reusing the model to score individual firms for grants or support eligibility would make it a tool for decisions about specific businesses and, through their owners, possibly about people. That use must be re-classified first, and the model card lists it as out of scope.

**The UK approach.** There is no AI statute: existing regulators apply five cross-sector principles, and the course asks for an artefact per principle, not a promise.

| Principle | Evidence in this project |
|---|---|
| Safety, security and robustness | Time-split test year never seen in tuning (tested); Event Backtests; monotone constraints; leakage tests; the API reports healthy only once artefacts are loaded |
| Transparency and explainability | The Why? page (exact SHAP from the serving model); this model card; the limitations panel; associational wording everywhere |
| Fairness | The bias review above, by region and income level; the region breakdown on the Model Evaluation page |
| Accountability and governance | Versioned artefacts with lineage; decisions recorded in ADRs and a decision log; the owner is still UNASSIGNED, which is the gap to close before use |
| Contestability and redress | Outputs are leads for Advisors, who decide; every Opportunity can be challenged through its explanation and Actual Exports history |

**Data protection.** The data are country-level trade aggregates with no personal data, so UK GDPR (including Article 22 on solely automated decisions) is not engaged and no DPIA is needed. As a tool that informs public-sector decisions, it would be a candidate for a record under the UK's Algorithmic Transparency Recording Standard, for which the model card is most of the content.

### Drift and the yearly refresh

A trained model is frozen while trade moves (course: *model drift and its three types*). For this model:

- **Data drift**, P(X): the inputs shift as economies grow and agreements change. The Population Stability Index (course: *PSI*) compares each feature's distribution on the UK grid in two years, rated stable below 0.10, watch to 0.20 and act above.
- **Concept drift**, P(y | X): the same fundamentals bring different trade, for example after the 2025 US tariff rises or new sanctions, none of which the model sees. It shows as falling accuracy while inputs look normal, so it needs the yearly re-evaluation, not PSI.
- **Label drift**, P(y): BACI revises its latest year, and the provisional 2024 already has 5.5% fewer Flows than 2023.

In [ ]:
if FIXTURE_MODE:
    last_year = feature_table.select(pl.col("year").max()).collect().item()
    drift_reference = feature_table.filter(pl.col("year") == last_year - 1).collect()
    drift_current = feature_table.filter(pl.col("year") == last_year).collect()
else:
    drift_reference = pl.read_parquet(feature_dir / "uk_grid_2023.parquet")
    drift_current = pl.read_parquet(feature_dir / "uk_grid_2024.parquet")
governance.feature_drift(drift_reference, drift_current, features.FEATURES)

On the real UK grid, 2023 against 2024, every Market-side feature is **stable** (all PSI below 0.01; the largest is the high-income share of imports at 0.008). Three features read **act**:

- **The UK's own GDP and population** (PSI 10.5) take a single value per year on the UK grid, so any change moves every row to a new bin. PSI is the wrong test for a constant; the right check is that the new value lies inside the clip range fitted on training years, which `FeatureTransform` enforces.
- **Relatedness** (PSI 0.24) is new in v2 and is a UK-side feature: it takes one value per Product, repeated across every Market, so a shift in the UK's export basket in the provisional 2024 data moves whole blocks of rows. It is not a broken input, but it is the one feature worth watching when the revised 2024 data arrive.

The policy features did not move, and the Applied Tariff cannot: it is carried forward from 2021 (ADR 0006), so PSI is blind to real tariff changes, which is a limitation of the data, not evidence of stability. The memberships will move in 2025, when the UK's CPTPP accession takes effect.

**The yearly refresh.** CEPII publishes a new BACI release each January, revising the provisional year and adding one more.

1. **Rebuild** from scratch with `scripts/rebuild.py` after updating `baci.BACI_VERSION` (and the HMRC year for the cross-check). The previous artefact stays in place until the new one is complete. Check whether TRAINS has published years after 2021; if it has, they replace the carried-forward tariffs without changing the feature (ADR 0006).
2. **Check inputs before training**: re-run the HMRC cross-check and the PSI table above. A large PSI on a Market-side feature is a question before it is a reason to retrain: a broken download or a changed code list must be fixed, not trained on (course: *responding to a drift alarm*).
3. **Retrain and re-evaluate** with the test year moved on by one. Promote the new version only if it beats the naive baseline on the headline metric (v2 does not yet, so this is the bar the next version has to clear), and compare the new Baseline's rankings with the old one (the stability check of section 7) so that Advisors are told when the top Opportunities move.
4. **Use the new year as a live test.** The UK's CPTPP accession counts from 2025, so the first BACI release with 2025 data turns the CPTPP Scenario into a real Event Backtest of a UK agreement.
5. **Roll back** by removing the newest artefact folder; the API then serves the previous version.

Each step needs a named owner, which, like the model's owner, is still UNASSIGNED.

**Conclusion.** The model is minimal risk under the EU AI Act for its intended purpose and fits the UK's principles with evidence for each, provided its use stays decision support for officials. Its main fairness risk is a data one: UK rankings are much weaker in low-income Markets, because BACI measures small Flows there least well, and Advisors are told so. The Potential Range holds its 80% target for the large Exporters most like the UK and is too wide, never too narrow, for small ones. The model card is generated from the artefact it describes: the Exporter scope, both sampling rates, the dropped feature and the Excluded Products are read from it, not typed in, and it states the v1 comparison, the PPML benchmark and the Potential Range's fixed dollar width as plainly as the results. The lineage now covers the 1,294 TRAINS files and the UNSD concordances. The open governance gaps are named rather than hidden: no owner, raw-file hashes not yet stored in the artefact, and no per-region Opportunity thresholds.

## 11. Conclusions

The project built an end-to-end system: data from primary sources (BACI, CEPII Gravity, WDI, IMF WEO, the WTO, UNCTAD TRAINS and HMRC) rebuilt by one script, a feature table shared by training and serving, five out-of-fold gradient-boosted models trained on the GPU, a FastAPI service, and a Streamlit app for Advisors. v2 changed the training rows, the features, the tuning, the Export Gap and the uncertainty, and then tested itself against v1 under a rule fixed in advance. This section sums up what the evidence supports, what goes beyond the course, and what to do next.

**What the evidence supports**

- **The model learns global trade well.** Across all Exporters, on 2023, a year it never saw, its within-Product rank correlation is 0.76 against 0.57 for the naive baseline, with about a twelfth of the baseline's deviance.
- **For UK pairs it does not beat a simple market-share rule on ranking** (0.51 against 0.52). Its value to Advisors is the much better estimate of the *size* of a Flow (a quarter of the baseline's deviance), with explanations: a map of where comparable exporters sell more, not a ranking that beats common sense.
- **v2 does not beat v1.** Under the pre-registered rule it ranks UK Markets less well (0.511 against 0.540, interval −0.029 to −0.027) and falls below the naive baseline that v1 beat, while estimating the size of Flows better (all-Exporter deviance 355 against 402). The gain ladder associates most of the drop (−0.020 of −0.028) with v2's new training rows, and about −0.004 each with the new features and the tuning. Every v2 choice was made on 2022 Tweedie deviance, which v2 improves; the UK ranking was only a guardrail. Nothing was retuned after 2023 was read.
- **Large Export Gaps concentrate later growth (Lift 5.2), but no more than other pairs of the same size (0.79).** An Opportunity marks a Market worth a look.
- **The model overstates what Trade Agreements are associated with.** After EU–Japan the model associated the agreement with +35% (EU to Japan) and +42% (Japan to the EU), against observed changes of +2% and −0%; for UK–Australia/New Zealand, +31% against −16%. A 3-way PPML gravity benchmark on the same Flows associates a Trade Agreement with +5.1% (−0.4% to +11.0%), against the model's +57% Baseline-weighted, and published estimates span about +21% to +100%. Scenario Effects indicate direction and an upper-end order of magnitude, never a forecast.
- **The Potential Range is well calibrated where it matters but coarse at the top.** Held-out coverage is 86% against 80%, close to 80% in every band above a few thousand dollars and for the large Exporters most like the UK. Its fixed dollar offsets give every pair above about $2.3m the same $22.9m-wide range, so for large pairs "Uncertain" only means an Export Gap under about $10.3m.
- **BACI's UK data differ from HMRC's by explained amounts**, and rank pairs alike (0.89).

**Beyond the course notes:** out-of-fold cross-fitting for the target country (ADR 0001); Tweedie loss with Horvitz–Thompson weighted zero sampling; monotone-constrained boosting, the Applied Tariff included, so that Scenarios respect the direction of policy (ADRs 0002 and 0006); a learning curve along the *data* axis under a VRAM budget (course: the bias–variance trade-off, read along rows rather than model complexity); Bayesian hyperparameter search with TPE and pruning (course: grid and randomised search); split conformal prediction with Mondrian bands, calibrated on other Exporters (course: held-out validation, extended to a range per prediction); a 3-way PPML gravity benchmark (course: linear regression, taken to Poisson pseudo-likelihood with fixed effects); a pre-registered comparison with a paired bootstrap and a gain ladder (course: touch the test set once); Lift with a cluster bootstrap and a size-matched control; difference-in-differences Event Backtests; an independent primary-source cross-check of the training data; and a bias review by reporting quality and by Potential Range coverage.

**Next steps**, in order of value:

1. **Win back the UK ranking.** Find which part of v2's new rows costs it (the top-100 scope, every UK Zero Cell at full weight, or the sampling rates) by changing one at a time on 2022, and choose with the UK ranking as an objective rather than a guardrail, so the loss and the metric Advisors read stop disagreeing (course: loss vs metric). Then the next test year decides.
2. **A hurdle model.** Most UK pairs have no Flow, and one Tweedie model fits whether trade happens and how large it is together. A hurdle model (a classifier for *any* Flow, then a model of size given a Flow) separates the two, which suits New Market and New Product Opportunities and gives a probability of trade that Advisors can read directly.
3. **An entity-embedding neural network.** Exporters, Importers and Products enter the trees through hand-built aggregates. A network that learns an embedding for each (course: embeddings, here for categories rather than words) could learn which Markets and Products behave alike directly from the Flows, and be compared with XGBoost on the same split and the same pre-registered rule. The course's advice that trees win on tabular data is the hypothesis to beat, not a reason to skip it.
4. **Calibrate Scenario Effects** against the PPML benchmark and a pool of Event Backtests, report a range rather than a point, and make the Potential Range relative to Export Potential so that it means something for large pairs.
5. **Train on every row** with XGBoost's external-memory mode, the last resort ADR 0004 keeps open, now that the learning curve showed the deviance still falling at the VRAM budget.
6. **Add sanctions, and tariffs after 2021** when a source allows, so that Russia-style Opportunities disappear and tariff Scenarios start from current rates.
7. **Name an owner**, store the lineage hashes in the artefact, and run the yearly refresh with the 2025 data as the first live test of the CPTPP Scenario.

**Conclusion.** The UK Exports Predictor is a transparent tool for finding *where to look*, built and evaluated so that its weaknesses are measured rather than hidden. v2 is the honest kind of second version: it fixed known problems (misleading Products, missing tariffs, one-year gaps, no sense of scatter), and under a rule set before the test year it did not beat v1 on the ranking that matters most, which this write-up reports rather than explains away. Its numbers should be read as associations learned from global trade: useful for prioritising conversations with industry, not for promising what a market or an agreement will deliver.